# V11 — Frozen DINOv3 ViT-7B with material-query patch attention

End-to-end Kaggle notebook for BDC Satria Data 2026. The notebook uses only official raw training images for development and selection, then unlocks the official test set once a signed train-only decision exists. The final path is one frozen backbone, one selected representation, one head, one original test view, and direct argmax.

**Before Run All:** select **GPU T4 x2**, enable internet, accept the gated DINOv3 license, add a Kaggle secret named `HF_TOKEN`, and set `TEAM_NAME` in the configuration cell. Large feature caches are resumable but excluded from the artifact bundle.

## Verified train-only evidence and failure hypothesis

| Experiment | Five-fold OOF Macro F1 | Errors | Recyclable↔Organic | Decision |
|---|---:|---:|---:|---|
| V7 final CLS + true-patch mean | 0.9901360067 | 287 | 241 | Historical strong baseline |
| V10 timm control | 0.9901662324 | 287 | 242 | Exact control reproduced below |
| V10 rich fixed spatial pooling, fold 0 | 0.9882367633 vs 0.9897706675 control | 72 vs 62 | 65 vs 55 | Rejected |
| V9 raw/drop/relabel, fold 0 | 0.989918 / 0.989471 / 0.989303 | 61 / 64 / 66 | 54 / 57 / 60 | Keep raw labels |
| V9A best cached recipe | 0.9904393208 vs 0.9903007998 control | — | 192 vs 188 on assessment rows | Rejected: +0.0001385, two fold wins, and a forbidden blend |

The remaining errors plausibly arise when global shape, color, or scene context dominates local material cues: flower-like paper/plastic, textiles and tissue, or mixed scenes. This is a **train-only modeling hypothesis**, not extra supervision and not a basis for row-level overrides.

Paper grounding is deliberately conservative. DINOv3 (`2508.10104v1`) directly supports strong normalized final-layer global and dense tokens; the semantic-selection study (`2602.07550v1`) shows that intermediate-layer heuristics are unreliable, supporting final-layer-only extraction. DINO-AugSeg (`2601.08078v1`) and the forensics study (`2604.16083v1`) support cross-attention or local patch readouts in other dense domains, but transfer to this classifier is an **untested hypothesis**. The industrial-waste review and optical-plastics study (`2505.16513v1`) directly warn that RGB models exploit color/context and cannot infer chemistry that is not visually observable. The TTA study (`2604.09697v1`) supports validating augmentations rather than assuming benefit. LoRA papers show domain-adaptation upside elsewhere, but do not establish safety or completion for a frozen 7B T4×2 protocol.

## Candidate decision record

| Candidate | Mechanism | Evidence / status | Upside | Compute/storage risk | Leakage risk | Decision |
|---|---|---|---|---|---|---|
| A. Four material queries over compressed 8×8 final patches | Preserve patch identity and learn relevant regions | DINOv3 dense tokens are strong; fixed V10 pooling failed; classification benefit untested | Directly targets local material/context failure | Low after fixed 256-D projection | Low with fixed projection and stable folds | **Implement** |
| B. Original + color-debiased + local-material consistency | Reduce color/shape shortcut while training one head | General augmentation evidence; must pass this dataset's gates | Complementary train-only regularization | Three projected views still fit budget | Low; original-only validation/test | **Integrate into A**, not a separate model |
| C. S2-FracMix/saliency mixing | Same-image salient patches may preserve labels | `2606.25784v1`; faithful method needs saliency/fractal machinery and new choices | Possible local robustness | High complexity and tuning surface | Moderate label-preservation risk | Reject |
| D. LoRA/adapters | Adapt backbone to domain | Positive in supplied medical/forensics papers | Potentially large | 7B backward pass risks T4 completion; violates frozen comparison | Low if train-only | Reject |
| E. Frequency/texture auxiliary branch | Emphasize texture | Indirect support only | Could help textile/paper | Adds another representation branch | Low | Reject as architecture soup |
| F. Local/global crop vector | Preserve a crop without attention | Simple fallback | Cheaper | Crop may omit target; less diagnostic | Low | Reject; A is affordable |

**Locked candidate:** final normalized true patches only, register tokens excluded; deterministic 2×2 average pooling (16×16→8×8); one persisted seeded Rademacher Johnson–Lindenstrauss projection (4096→256); four learned queries; one cross-attention block; projected CLS and true-patch mean; one compact classifier. No intermediate layers, branch zoo, blend, TTA, routing, or post-hoc correction.

## Locked resources, views, loss, and gates

Each projected view stores 66×256 FP16 values (64 spatial tokens + CLS + patch mean): about **0.835 GiB** for 26,527 rows. Three views are about **2.504 GiB**; the exact 8,192-D FP16 original-view control adds about **0.405 GiB**; total train cache is about **2.91 GiB**. The larger possible selected test cache adds only about **0.046 GiB**, keeping all feature caches below the locked 8 GiB limit. A candidate batch of 256 with three FP32 views is about 49.5 MiB before head activations. Three frozen forwards per train row cost roughly three times V10 extraction; the 4-image feature batch is retained. Preflight measures current Kaggle working-disk availability after checkpoint download and requires at least 2 GiB to remain after the projected caches.

Views are deterministic per image and seed: exact original; a geometry-preserving 65% grayscale blend with mild contrast (`color_debiased`); and a conservative seeded crop (`scale 0.45–1.00`, ratio `0.75–1.33`) with mild contrast (`local_material`). The loss is `CE(original) + 0.5 CE(color) + 0.5 CE(local) + 0.2 mean(JS(original,color), JS(original,local))`, with class weights and label smoothing 0.05. Validation/test use original only.

Fold-0 gate: Macro F1 gain ≥0.0010, errors lower by ≥5, Recyclable↔Organic lower by ≥5, Electronic F1 decline ≤0.0005, finite/order checks, and intact single-model contract. Only a full pass unlocks candidate folds 1–4. Adoption additionally requires OOF Macro F1 ≥0.9911, gain ≥0.0010, errors ≤282, Recyclable↔Organic ≤237, ≥3 fold wins, no more than 75% of positive gain from one fold, stable Electronic F1, and no leakage/overrides. Gates are encoded below and cannot be weakened after results.

## 1. Kaggle dependencies

In [ ]:
import subprocess
import sys

subprocess.check_call([
    sys.executable, '-m', 'pip', 'install', '-q',
    'timm==1.0.20',
    'accelerate==1.10.1',
    'safetensors==0.6.2',
    'huggingface_hub==0.34.4',
])


## 2. Imports and immutable experiment configuration

In [ ]:
import gc
import hashlib
import json
import math
import os
import platform
import random
import shutil
import statistics
import time
import zipfile
from pathlib import Path

os.environ['TOKENIZERS_PARALLELISM'] = 'false'
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')
os.environ.setdefault('CUBLAS_WORKSPACE_CONFIG', ':4096:8')

import accelerate
import numpy as np
import pandas as pd
import PIL
import safetensors
import sklearn
import timm
import timm.data
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
from PIL import Image, ImageFile, ImageOps
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score, log_loss
from sklearn.model_selection import StratifiedKFold
from torch.utils.data import DataLoader, Dataset
from torchvision.transforms import InterpolationMode, v2
from torchvision.transforms import functional as TVF
from tqdm.auto import tqdm

# USER CONFIGURATION
TEAM_NAME = 'CHANGE_ME'
FEATURE_BATCH_SIZE = 4  # Lower only for post-load extraction OOM; it cannot fix weight-loading OOM.
FEATURE_LOADER_WORKERS = 2

# EXPERIMENT IDENTITY
EXPERIMENT_NAME = 'v11_dinov3_vit7b_material_aware_patch_attention'
REVISION_MARKER = 'v11_material_query_jl256_three_view_seed42_v1'
TIMM_MODEL_NAME = 'vit_7b_patch16_dinov3.lvd1689m'
MODEL_REPO_ID = 'timm/vit_7b_patch16_dinov3.lvd1689m'
MODEL_REVISION_REQUEST = 'main'

# VERIFIED TOKEN/PREPROCESSING CONTRACT
IMAGE_SIZE = 256
PATCH_SIZE = 16
PATCH_GRID = 16
POOLED_GRID = 8
EXPECTED_BLOCKS = 40
EXPECTED_HIDDEN_SIZE = 4096
EXPECTED_PATCH_TOKENS = 256
EXPECTED_REGISTER_TOKENS = 4
EXPECTED_PREFIX_TOKENS = 5
EXPECTED_SEQUENCE_TOKENS = 261
BACKBONE_DTYPE = torch.float16
FEATURE_DTYPE = torch.float16
IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)
GPU_STATIC_LIMIT_GIB = 10.5
GPU_MIN_HEADROOM_GIB = 3.0
MIN_EXTRACTION_IMAGES_PER_SECOND = 0.8

# FIXED REPRESENTATION
PROJECTION_DIM = 256
PROJECTION_SEED = 20260728
SPATIAL_TOKENS = 64
GLOBAL_TOKENS = 2
CANDIDATE_TOKENS = SPATIAL_TOKENS + GLOBAL_TOKENS
CONTROL_DIM = 2 * EXPECTED_HIDDEN_SIZE
VIEWS = ('original', 'color_debiased', 'local_material')
FEATURE_SHARD_SIZE = 256
TRAIN_CACHE_BUDGET_GIB = 8.0
MIN_WORKING_DISK_MARGIN_GIB = 2.0

# OFFICIAL DATA CONTRACT
CLASS_DIR_TO_LABEL = {'0_Recyclable': 0, '1_Electronic': 1, '2_Organic': 2}
CLASS_NAMES = ('Recyclable', 'Electronic', 'Organic')
NUM_CLASSES = 3
EXPECTED_RAW_COUNTS = {0: 9999, 1: 3961, 2: 12567}
EXPECTED_TRAIN_ROWS = 26527
EXPECTED_TEST_ROWS = 1458
SUBMISSION_COLUMNS = ('id', 'predicted')
ALLOWED_IMAGE_SUFFIXES = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}

# STABLE TRAINING PROTOCOL
SEED = 42
N_FOLDS = 5
DEVELOPMENT_FOLD = 0
ASSESSMENT_FOLDS = (1, 2, 3, 4)
V10_FOLD_ASSIGNMENT_SHA256 = '0a06976e5dc3cdb0b3f1f5043ff033bf95a40dd2b0cb82154bbf34d701ee305c'
CONTROL_NAME = 'timm_control'
CANDIDATE_NAME = 'material_query_attention'
MAX_EPOCHS = 15
EARLY_STOPPING_PATIENCE = 5
HEAD_BATCH_SIZE = 256
PREDICT_BATCH_SIZE = 512
LEARNING_RATE = 8e-4
WEIGHT_DECAY = 1e-4
LABEL_SMOOTHING = 0.05
GRAD_CLIP_MAX_NORM = 1.0
CONTROL_HIDDEN_DIM = 512
HEAD_DROPOUT = 0.40
MATERIAL_QUERIES = 4
ATTENTION_HEADS = 8
MATERIAL_FFN_DIM = 512
VIEW_CE_WEIGHTS = {'original': 1.0, 'color_debiased': 0.5, 'local_material': 0.5}
CONSISTENCY_WEIGHT = 0.2

# PRELOCKED DEVELOPMENT AND ADOPTION GATES
DEV_MIN_GAIN = 0.0010
DEV_MIN_ERROR_REDUCTION = 5
DEV_MIN_RO_REDUCTION = 5
MAX_ELECTRONIC_F1_DROP = 0.0005
ADOPT_MIN_OOF_F1 = 0.9911
ADOPT_MIN_GAIN = 0.0010
ADOPT_MAX_ERRORS = 282
ADOPT_MAX_RO_ERRORS = 237
ADOPT_MIN_FOLD_WINS = 3
ADOPT_MAX_SINGLE_FOLD_GAIN_SHARE = 0.75

V7_REFERENCE = {
    'oof_macro_f1': 0.9901360067264733,
    'total_errors': 287,
    'recyclable_organic_errors': 241,
}
V10_REFERENCE = {
    'fold0_control_macro_f1': 0.9897706675032462,
    'fold0_control_errors': 62,
    'fold0_control_recyclable_organic_errors': 55,
    'fold0_rich_macro_f1': 0.9882367633437932,
    'oof_macro_f1': 0.9901662323927614,
    'total_errors': 287,
    'recyclable_organic_errors': 242,
}

assert PATCH_GRID * PATCH_GRID == EXPECTED_PATCH_TOKENS
assert POOLED_GRID * POOLED_GRID == SPATIAL_TOKENS
assert PROJECTION_DIM % ATTENTION_HEADS == 0
assert VIEWS[0] == 'original' and len(VIEWS) == 3
assert sum(EXPECTED_RAW_COUNTS.values()) == EXPECTED_TRAIN_ROWS
assert SEED == 42 and ASSESSMENT_FOLDS == (1, 2, 3, 4)

TEST_ACCESS_STATE = {'unlocked': False, 'test_images_accessed': False, 'manual_test_labels_accessed': False}
NOTEBOOK_STARTED_AT = time.perf_counter()
print('Experiment:', EXPERIMENT_NAME)
if TEAM_NAME == 'CHANGE_ME':
    print('ACTION REQUIRED: set TEAM_NAME before submission.')


## 3. Atomic artifacts, deterministic runtime, and two-T4/test-isolation guards

In [ ]:
def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def json_default(value):
    if isinstance(value, (np.integer, np.floating, np.bool_)):
        return value.item()
    if isinstance(value, np.ndarray):
        return value.tolist()
    if isinstance(value, Path):
        return str(value)
    raise TypeError('Not JSON serializable: {}'.format(type(value).__name__))


def sha256_json(value):
    payload = json.dumps(
        value, sort_keys=True, separators=(',', ':'), ensure_ascii=False, default=json_default, allow_nan=False
    )
    return hashlib.sha256(payload.encode('utf-8')).hexdigest()


def sha256_file(path, chunk_bytes=8 * 1024 * 1024):
    digest = hashlib.sha256()
    with Path(path).open('rb') as handle:
        while True:
            chunk = handle.read(chunk_bytes)
            if not chunk:
                break
            digest.update(chunk)
    return digest.hexdigest()


def write_json_atomic(value, path):
    path = Path(path)
    temporary = path.with_name(path.name + '.tmp')
    with temporary.open('w', encoding='utf-8') as handle:
        json.dump(value, handle, indent=2, sort_keys=True, default=json_default, allow_nan=False)
        handle.flush()
        os.fsync(handle.fileno())
    os.replace(temporary, path)


def write_csv_atomic(frame, path, index=False):
    path = Path(path)
    temporary = path.with_name(path.name + '.tmp')
    frame.to_csv(temporary, index=index)
    os.replace(temporary, path)


def write_npy_atomic(array, path):
    path = Path(path)
    temporary = path.with_name(path.name + '.tmp.npy')
    np.save(temporary, array)
    os.replace(temporary, path)


def atomic_torch_save(value, path):
    path = Path(path)
    temporary = path.with_name(path.name + '.tmp')
    with temporary.open('wb') as handle:
        torch.save(value, handle)
        handle.flush()
        os.fsync(handle.fileno())
    os.replace(temporary, path)


def read_json(path):
    with Path(path).open('r', encoding='utf-8') as handle:
        return json.load(handle)


def cuda_memory_snapshot(label):
    rows = []
    for device_id in range(torch.cuda.device_count()):
        free_bytes, total_bytes = torch.cuda.mem_get_info(device_id)
        rows.append({
            'label': label,
            'device_id': device_id,
            'free_gib': free_bytes / 2 ** 30,
            'total_gib': total_bytes / 2 ** 30,
            'allocated_gib': torch.cuda.memory_allocated(device_id) / 2 ** 30,
            'reserved_gib': torch.cuda.memory_reserved(device_id) / 2 ** 30,
            'peak_allocated_gib': torch.cuda.max_memory_allocated(device_id) / 2 ** 30,
        })
    return rows


def release_cuda_cache():
    gc.collect()
    for device_id in range(torch.cuda.device_count()):
        with torch.cuda.device(device_id):
            torch.cuda.empty_cache()


seed_everything(SEED)
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True
torch.use_deterministic_algorithms(True)
torch.set_float32_matmul_precision('high')
Image.MAX_IMAGE_PIXELS = 200_000_000
ImageFile.LOAD_TRUNCATED_IMAGES = False

KAGGLE_WORKING = Path('/kaggle/working')
KAGGLE_INPUT = Path('/kaggle/input')
if not KAGGLE_WORKING.is_dir() or not KAGGLE_INPUT.is_dir():
    raise RuntimeError('Run this notebook on Kaggle with the official dataset attached.')

ARTIFACT_DIR = KAGGLE_WORKING / EXPERIMENT_NAME
CACHE_DIR = ARTIFACT_DIR / 'feature_cache'
TRAIN_CACHE_DIR = CACHE_DIR / 'train'
TEST_CACHE_DIR = CACHE_DIR / 'test'
HEADS_DIR = ARTIFACT_DIR / 'heads'
for directory in (ARTIFACT_DIR, CACHE_DIR, TRAIN_CACHE_DIR, TEST_CACHE_DIR, HEADS_DIR):
    directory.mkdir(parents=True, exist_ok=True)

SELECTION_DECISION_PATH = ARTIFACT_DIR / 'selection_decision.json'
PREFLIGHT_REPORT_PATH = ARTIFACT_DIR / 'preflight_report.json'
PROJECTION_STATE_PATH = ARTIFACT_DIR / 'projection_state.pt'

if not torch.cuda.is_available() or torch.cuda.device_count() != 2:
    raise RuntimeError('Exactly two CUDA devices are required; choose Kaggle GPU T4 x2.')
GPU_RUNTIME_INFO = []
for device_id in (0, 1):
    properties = torch.cuda.get_device_properties(device_id)
    if 'T4' not in properties.name.upper():
        raise RuntimeError('cuda:{} is {}, not a Tesla T4.'.format(device_id, properties.name))
    GPU_RUNTIME_INFO.append({
        'device_id': device_id,
        'name': properties.name,
        'total_vram_gib': properties.total_memory / 2 ** 30,
    })

suspicious_label_files = []
for path in KAGGLE_INPUT.rglob('*.csv'):
    lowered = path.name.lower().replace('-', '_')
    label_like = any(token in lowered for token in ('label', 'truth', 'annotation'))
    test_like = any(token in lowered for token in ('test', 'manual', 'ground'))
    if label_like and test_like:
        suspicious_label_files.append(str(path))
if suspicious_label_files:
    raise RuntimeError('Detach manual/test-label inputs before running:\n' + '\n'.join(suspicious_label_files))

ENVIRONMENT = {
    'python': platform.python_version(),
    'platform': platform.platform(),
    'torch': torch.__version__,
    'cuda': torch.version.cuda,
    'cudnn': torch.backends.cudnn.version(),
    'timm': timm.__version__,
    'accelerate': accelerate.__version__,
    'safetensors': safetensors.__version__,
    'numpy': np.__version__,
    'pandas': pd.__version__,
    'pillow': PIL.__version__,
    'scikit_learn': sklearn.__version__,
    'torchvision': torchvision.__version__,
    'cublas_workspace_config': os.environ.get('CUBLAS_WORKSPACE_CONFIG'),
    'deterministic_algorithms': True,
    'gpus': GPU_RUNTIME_INFO,
}
CONFIG = {
    'experiment_name': EXPERIMENT_NAME,
    'revision_marker': REVISION_MARKER,
    'team_name': TEAM_NAME,
    'seed': SEED,
    'model_repo': MODEL_REPO_ID,
    'timm_model': TIMM_MODEL_NAME,
    'image_size': IMAGE_SIZE,
    'token_contract': {
        'prefix': EXPECTED_PREFIX_TOKENS,
        'registers_excluded': EXPECTED_REGISTER_TOKENS,
        'true_patches': EXPECTED_PATCH_TOKENS,
        'hidden': EXPECTED_HIDDEN_SIZE,
        'final_norm_applied_times': 1,
    },
    'projection': {'kind': 'seeded_rademacher_jl', 'seed': PROJECTION_SEED, 'output_dim': PROJECTION_DIM},
    'views': list(VIEWS),
    'loss': {'view_ce_weights': VIEW_CE_WEIGHTS, 'consistency_weight': CONSISTENCY_WEIGHT},
    'training': {
        'max_epochs': MAX_EPOCHS,
        'patience': EARLY_STOPPING_PATIENCE,
        'batch_size': HEAD_BATCH_SIZE,
        'learning_rate': LEARNING_RATE,
        'weight_decay': WEIGHT_DECAY,
        'label_smoothing': LABEL_SMOOTHING,
        'gradient_clip': GRAD_CLIP_MAX_NORM,
    },
    'development_gate': {
        'min_macro_f1_gain': DEV_MIN_GAIN,
        'min_error_reduction': DEV_MIN_ERROR_REDUCTION,
        'min_ro_error_reduction': DEV_MIN_RO_REDUCTION,
        'max_electronic_f1_drop': MAX_ELECTRONIC_F1_DROP,
    },
    'adoption_gate': {
        'min_oof_macro_f1': ADOPT_MIN_OOF_F1,
        'min_gain': ADOPT_MIN_GAIN,
        'max_errors': ADOPT_MAX_ERRORS,
        'max_ro_errors': ADOPT_MAX_RO_ERRORS,
        'min_fold_wins': ADOPT_MIN_FOLD_WINS,
        'max_single_fold_gain_share': ADOPT_MAX_SINGLE_FOLD_GAIN_SHARE,
        'max_electronic_f1_drop': MAX_ELECTRONIC_F1_DROP,
    },
    'single_model_contract': True,
    'test_view': 'original',
    'tta': False,
    'ensemble_or_blend': False,
}
# TEAM_NAME changes only the submission filename, never the locked modeling recipe.
CONFIG_SIGNATURE = sha256_json({key: value for key, value in CONFIG.items() if key != 'team_name'})
CONFIG['modeling_config_signature'] = CONFIG_SIGNATURE
write_json_atomic(CONFIG, ARTIFACT_DIR / 'config.json')
write_json_atomic(ENVIRONMENT, ARTIFACT_DIR / 'environment.json')
print('Artifact directory:', ARTIFACT_DIR)
print('Configuration signature:', CONFIG_SIGNATURE)
print('GPU contract:', GPU_RUNTIME_INFO)


## 4. Pinned checkpoint, raw-train discovery, deterministic views, and stable folds

In [ ]:
from huggingface_hub import model_info, snapshot_download


def resolve_hf_token():
    token = os.environ.get('HF_TOKEN')
    if not token:
        try:
            from kaggle_secrets import UserSecretsClient
            token = UserSecretsClient().get_secret('HF_TOKEN')
        except Exception:
            token = None
    if not token:
        raise RuntimeError(
            'Add a Kaggle Secret named HF_TOKEN, accept the DINOv3 license for {}, and enable internet.'.format(
                MODEL_REPO_ID
            )
        )
    return token


HF_TOKEN_VALUE = resolve_hf_token()
try:
    RESOLVED_REVISION = model_info(
        MODEL_REPO_ID, revision=MODEL_REVISION_REQUEST, token=HF_TOKEN_VALUE
    ).sha
except Exception as exc:
    raise RuntimeError('Could not resolve the gated model revision. Check token, license, and internet.') from exc
if len(RESOLVED_REVISION or '') != 40 or any(c not in '0123456789abcdef' for c in RESOLVED_REVISION.lower()):
    raise RuntimeError('Hugging Face did not return an immutable 40-character commit SHA.')

download_started = time.perf_counter()
MODEL_SNAPSHOT_DIR = Path(snapshot_download(
    repo_id=MODEL_REPO_ID,
    revision=RESOLVED_REVISION,
    token=HF_TOKEN_VALUE,
    allow_patterns=['*.safetensors', '*.safetensors.index.json', '*.json'],
))
MODEL_DOWNLOAD_SECONDS = time.perf_counter() - download_started
del HF_TOKEN_VALUE


def discover_named_train_dirs(base, max_depth=3):
    found = []
    stack = [(base, 0)]
    while stack:
        current, depth = stack.pop()
        try:
            children = list(current.iterdir())
        except OSError:
            continue
        for child in children:
            if not child.is_dir():
                continue
            lowered = child.name.lower()
            if lowered == 'test':
                continue
            if lowered == 'train':
                found.append(child)
            elif depth < max_depth:
                stack.append((child, depth + 1))
    return found


def exact_raw_train_signature(train_dir):
    try:
        roots = list(train_dir.iterdir())
        if {path.name for path in roots} != set(CLASS_DIR_TO_LABEL):
            return None
        if any(path.is_symlink() or not path.is_dir() for path in roots):
            return None
        counts = {}
        for directory_name, label in CLASS_DIR_TO_LABEL.items():
            entries = list((train_dir / directory_name).iterdir())
            if any(
                path.is_symlink() or not path.is_file() or path.suffix.lower() not in ALLOWED_IMAGE_SUFFIXES
                for path in entries
            ):
                return None
            counts[label] = len(entries)
        return counts
    except OSError:
        return None


def resolve_train_dir():
    candidates = discover_named_train_dirs(KAGGLE_INPUT)
    matches = {path.resolve() for path in candidates if exact_raw_train_signature(path) == EXPECTED_RAW_COUNTS}
    if len(matches) != 1:
        raise RuntimeError('Expected exactly one official raw train directory; found {}: {}'.format(
            len(matches), sorted(map(str, matches))
        ))
    return matches.pop()


def load_rgb_image(path):
    try:
        with Image.open(path) as image:
            width, height = image.size
            if width <= 0 or height <= 0:
                raise ValueError('invalid dimensions')
            needs_downscale = width * height >= 8_000_000 or max(width, height) > 4096
            image.draft('RGB', (1024, 1024))
            rgb = ImageOps.exif_transpose(image).convert('RGB')
            if needs_downscale:
                rgb.thumbnail((1024, 1024), Image.Resampling.LANCZOS)
            return rgb.copy()
    except Exception as exc:
        raise RuntimeError('Cannot decode image: {}'.format(path)) from exc


to_resized_float = v2.Compose([
    v2.ToImage(),
    v2.Resize((IMAGE_SIZE, IMAGE_SIZE), interpolation=InterpolationMode.BILINEAR, antialias=True),
    v2.ToDtype(torch.float32, scale=True),
])


def normalize_image(tensor):
    return TVF.normalize(tensor, IMAGENET_MEAN, IMAGENET_STD)


def stable_view_generator(image_id, view_name):
    digest = hashlib.sha256('{}|{}|{}'.format(SEED, image_id, view_name).encode('utf-8')).digest()
    return torch.Generator().manual_seed(int.from_bytes(digest[:8], 'little') % (2 ** 63 - 1))


def deterministic_crop(image, image_id):
    generator = stable_view_generator(image_id, 'local_material')
    width, height = image.size
    area = width * height
    log_ratio = (math.log(0.75), math.log(1.33))
    for _ in range(10):
        scale = 0.45 + 0.55 * torch.rand((), generator=generator).item()
        ratio = math.exp(log_ratio[0] + (log_ratio[1] - log_ratio[0]) * torch.rand((), generator=generator).item())
        crop_width = int(round(math.sqrt(area * scale * ratio)))
        crop_height = int(round(math.sqrt(area * scale / ratio)))
        if 0 < crop_width <= width and 0 < crop_height <= height:
            left = int(torch.randint(0, width - crop_width + 1, (), generator=generator).item())
            top = int(torch.randint(0, height - crop_height + 1, (), generator=generator).item())
            return TVF.crop(image, top, left, crop_height, crop_width), generator
    side = min(width, height)
    left = (width - side) // 2
    top = (height - side) // 2
    return TVF.crop(image, top, left, side, side), generator


def make_view(image, image_id, view_name):
    if view_name == 'original':
        tensor = to_resized_float(image)
    elif view_name == 'color_debiased':
        generator = stable_view_generator(image_id, view_name)
        tensor = to_resized_float(image)
        gray = TVF.rgb_to_grayscale(tensor, num_output_channels=3)
        tensor = 0.35 * tensor + 0.65 * gray
        contrast = 0.85 + 0.30 * torch.rand((), generator=generator).item()
        tensor = TVF.adjust_contrast(tensor, contrast).clamp(0.0, 1.0)
    elif view_name == 'local_material':
        cropped, generator = deterministic_crop(image, image_id)
        tensor = to_resized_float(cropped)
        contrast = 0.90 + 0.20 * torch.rand((), generator=generator).item()
        tensor = TVF.adjust_contrast(tensor, contrast).clamp(0.0, 1.0)
    else:
        raise KeyError('Unknown train view: {}'.format(view_name))
    output = normalize_image(tensor)
    if tuple(output.shape) != (3, IMAGE_SIZE, IMAGE_SIZE) or not torch.isfinite(output).all():
        raise RuntimeError('Invalid transformed image for {}.'.format(image_id))
    return output


PREPROCESS_CONFIG = {
    'decoder': 'PIL with EXIF transpose and RGB conversion',
    'large_image_guard': 'draft plus LANCZOS thumbnail to max side 1024 when >=8MP or >4096px',
    'resize': [IMAGE_SIZE, IMAGE_SIZE],
    'geometry': 'direct square, no crop for original',
    'interpolation': 'bilinear antialiased',
    'rescale': 'uint8 to float32 [0,1]',
    'normalization': {'mean': list(IMAGENET_MEAN), 'std': list(IMAGENET_STD)},
    'native_v10_parity': True,
}
PREPROCESS_SIGNATURE = sha256_json(PREPROCESS_CONFIG)
VIEW_CONFIG = {
    'original': {'geometry': 'direct square 256', 'interpolation': 'bilinear', 'augmentation': None},
    'color_debiased': {'grayscale_blend': 0.65, 'contrast': [0.85, 1.15], 'geometry': 'unchanged'},
    'local_material': {'crop_scale': [0.45, 1.0], 'crop_ratio': [0.75, 1.33], 'contrast': [0.9, 1.1]},
    'preprocess_signature': PREPROCESS_SIGNATURE,
    'validation_view': 'original',
    'test_view': 'original',
    'seed_derivation': 'sha256(global seed | relative image id | view name)',
}
VIEW_SIGNATURE = sha256_json(VIEW_CONFIG)


def index_raw_train(train_dir):
    rows = []
    for directory_name, label in CLASS_DIR_TO_LABEL.items():
        directory = train_dir / directory_name
        entries = sorted(directory.iterdir(), key=lambda path: path.name)
        for path in entries:
            if path.is_symlink() or not path.is_file() or path.suffix.lower() not in ALLOWED_IMAGE_SUFFIXES:
                raise RuntimeError('Invalid entry in official train directory: {}'.format(path))
            rows.append({
                'image_id': path.relative_to(train_dir).as_posix(),
                'path': str(path.resolve()),
                'label': label,
                'class_name': CLASS_NAMES[label],
                'file_size_bytes': path.stat().st_size,
            })
    frame = pd.DataFrame(rows).sort_values('image_id').reset_index(drop=True)
    if len(frame) != EXPECTED_TRAIN_ROWS or not frame['image_id'].is_unique:
        raise RuntimeError('Official train row count or identity uniqueness failed.')
    if frame['label'].value_counts().sort_index().to_dict() != EXPECTED_RAW_COUNTS:
        raise RuntimeError('Official train class counts failed.')
    return frame


TRAIN_DIR = resolve_train_dir()
train_df = index_raw_train(TRAIN_DIR)
train_df['fold'] = -1
splitter = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
for fold, (_, valid_indices) in enumerate(splitter.split(train_df, train_df['label'])):
    train_df.loc[valid_indices, 'fold'] = fold
train_df['fold'] = train_df['fold'].astype(np.int64)
if set(train_df['fold']) != set(range(N_FOLDS)):
    raise RuntimeError('Stable fold assignment is incomplete.')

ORDERED_IMAGE_IDS_SIGNATURE = sha256_json(train_df['image_id'].tolist())
LABEL_SIGNATURE = sha256_json(train_df['label'].astype(int).tolist())
FOLD_ASSIGNMENT_SIGNATURE = sha256_json(train_df[['image_id', 'label', 'fold']].to_dict('records'))
if FOLD_ASSIGNMENT_SIGNATURE != V10_FOLD_ASSIGNMENT_SHA256:
    raise RuntimeError('Recomputed folds do not match the persisted V10 stable fold signature.')
SOURCE_IDENTITY_SIGNATURE = sha256_json(
    train_df[['image_id', 'label', 'file_size_bytes']].to_dict('records')
)
FOLD_SIGNATURE = {
    'implementation': 'StratifiedKFold(n_splits=5, shuffle=True, random_state=42)',
    'ordering': 'relative image_id ascending before split',
    'development_fold': DEVELOPMENT_FOLD,
    'assessment_folds': list(ASSESSMENT_FOLDS),
    'fold_assignment_sha256': FOLD_ASSIGNMENT_SIGNATURE,
    'matches_persisted_v10_signature': True,
    'class_counts_by_fold': train_df.groupby(['fold', 'label']).size().unstack(fill_value=0).to_dict('index'),
}
DATA_CONTRACT = {
    'canonical_data': 'official raw labeled train only',
    'train_root': str(TRAIN_DIR),
    'rows': len(train_df),
    'class_mapping': {str(index): name for index, name in enumerate(CLASS_NAMES)},
    'raw_counts': EXPECTED_RAW_COUNTS,
    'ordered_image_ids_sha256': ORDERED_IMAGE_IDS_SIGNATURE,
    'labels_sha256': LABEL_SIGNATURE,
    'source_identity_sha256': SOURCE_IDENTITY_SIGNATURE,
    'fold_signature': FOLD_SIGNATURE,
    'preprocess_config': PREPROCESS_CONFIG,
    'preprocess_signature': PREPROCESS_SIGNATURE,
    'view_signature': VIEW_SIGNATURE,
    'external_images_or_labels': False,
    'dropped_or_relabeled_rows': 0,
    'test_images_accessed': False,
}
DATA_CONTRACT_SIGNATURE = sha256_json(DATA_CONTRACT)
write_csv_atomic(train_df[['image_id', 'label', 'class_name', 'fold']], ARTIFACT_DIR / 'fold_assignments.csv')
write_json_atomic(FOLD_SIGNATURE, ARTIFACT_DIR / 'fold_signature.json')
write_json_atomic(DATA_CONTRACT, ARTIFACT_DIR / 'data_contract.json')
print('Pinned revision:', RESOLVED_REVISION)
print('Official train:', TRAIN_DIR, '| rows:', len(train_df))
print('Fold signature:', FOLD_ASSIGNMENT_SIGNATURE)


## 5. Explicit streamed ViT-7B placement and final-token contract

The loader follows the proven V10 path: instantiate on `meta`, stream each safetensors tensor on CPU, cast floating weights to FP16 before GPU transfer, and dispatch an explicit fixed split (blocks 0–19 on CUDA 0; 20–39 on CUDA 1). Automatic maps and CPU/disk offload are forbidden. The runtime audit derives and checks the actual model structure rather than accepting hard-coded output claims.

In [ ]:
from accelerate import dispatch_model, init_empty_weights
from accelerate.utils import set_module_tensor_to_device
from safetensors import safe_open

OUTPUT_SIDE_MODULES = {'norm', 'fc_norm', 'head_norm', 'attn_pool', 'head_drop', 'head', 'head_dist'}


def normalized_device(value):
    if isinstance(value, int):
        return 'cuda:{}'.format(value)
    text = str(value).lower()
    return 'cuda:' + text if text in {'0', '1'} else text


def tensor_static_bytes(tensor):
    element_bytes = 2 if tensor.is_floating_point() else tensor.element_size()
    return tensor.numel() * element_bytes


def module_static_bytes(module):
    total = sum(tensor_static_bytes(parameter) for parameter in module.parameters(recurse=True))
    for _, child in module.named_modules():
        for name, buffer in child._buffers.items():
            if buffer is not None and name not in child._non_persistent_buffers_set:
                total += tensor_static_bytes(buffer)
    return total


def root_tensors(model):
    rows = []
    for name, tensor in list(model._parameters.items()) + list(model._buffers.items()):
        if tensor is not None and name not in model._non_persistent_buffers_set:
            rows.append((name, tensor_static_bytes(tensor)))
    return rows


def build_fixed_device_plan(model):
    if not hasattr(model, 'blocks') or len(model.blocks) != EXPECTED_BLOCKS:
        raise RuntimeError('Expected exactly 40 ordered transformer blocks.')
    root_rows = root_tensors(model)
    input_modules = []
    output_modules = []
    for name, module in model.named_children():
        if name == 'blocks':
            continue
        target = output_modules if name in OUTPUT_SIDE_MODULES else input_modules
        target.append((name, module_static_bytes(module)))
    block_bytes = [module_static_bytes(block) for block in model.blocks]
    device_map = {name: 'cuda:0' for name, _ in root_rows}
    device_map.update({name: 'cuda:0' for name, _ in input_modules})
    device_map.update({'blocks.{}'.format(index): 'cuda:0' if index < 20 else 'cuda:1' for index in range(40)})
    device_map.update({name: 'cuda:1' for name, _ in output_modules})
    gpu0_bytes = sum(size for _, size in root_rows + input_modules) + sum(block_bytes[:20])
    gpu1_bytes = sum(size for _, size in output_modules) + sum(block_bytes[20:])
    plan = {
        'strategy': 'explicit fixed contiguous streamed FP16',
        'automatic_device_map_used': False,
        'cpu_or_disk_offload': False,
        'block_count': 40,
        'cuda_0_blocks': [0, 19],
        'cuda_1_blocks': [20, 39],
        'input_side_modules': [name for name, _ in input_modules],
        'output_side_modules': [name for name, _ in output_modules],
        'root_tensors': [name for name, _ in root_rows],
        'estimated_static_gib': {'cuda:0': gpu0_bytes / 2 ** 30, 'cuda:1': gpu1_bytes / 2 ** 30},
        'device_map': device_map,
    }
    write_json_atomic(plan, ARTIFACT_DIR / 'planned_device_map.json')
    for device_id in (0, 1):
        estimate = plan['estimated_static_gib']['cuda:{}'.format(device_id)]
        _free, total = torch.cuda.mem_get_info(device_id)
        headroom = total / 2 ** 30 - estimate
        if estimate > GPU_STATIC_LIMIT_GIB or headroom < GPU_MIN_HEADROOM_GIB:
            raise RuntimeError(
                'Fixed FP16 placement leaves insufficient cuda:{} headroom ({:.2f} GiB). Restart T4 x2; '
                'lowering feature batch size cannot fix model-loading OOM.'.format(device_id, headroom)
            )
    return plan


def resolve_checkpoint_files(snapshot_dir):
    index_path = snapshot_dir / 'model.safetensors.index.json'
    if index_path.is_file():
        names = sorted(set(read_json(index_path)['weight_map'].values()))
    else:
        available = sorted(snapshot_dir.glob('*.safetensors'))
        if len(available) != 1:
            raise RuntimeError('Expected one safetensors file or an unambiguous index; found {}.'.format(available))
        names = [available[0].name]
    files = []
    for name in names:
        path = snapshot_dir / name
        if path.parent != snapshot_dir or '/' in name or not path.is_file():
            raise RuntimeError('Unsafe or missing checkpoint entry: {}'.format(name))
        files.append(path)
    total_gib = sum(path.stat().st_size for path in files) / 2 ** 30
    if not 1.0 <= total_gib <= 60.0:
        raise RuntimeError('Implausible checkpoint size: {:.2f} GiB.'.format(total_gib))
    return files, total_gib


def device_for_tensor(tensor_name, device_map):
    matches = [
        (len(module_name), device)
        for module_name, device in device_map.items()
        if tensor_name == module_name or tensor_name.startswith(module_name + '.')
    ]
    if not matches:
        raise RuntimeError('No planned device covers checkpoint tensor {}.'.format(tensor_name))
    return max(matches)[1]


def stream_checkpoint(model, checkpoint_files, device_map):
    model_names = set(model.state_dict())
    checkpoint_names = set()
    loaded = 0
    for checkpoint_file in checkpoint_files:
        with safe_open(str(checkpoint_file), framework='pt', device='cpu') as handle:
            for key in tqdm(sorted(handle.keys()), desc='streaming {}'.format(checkpoint_file.name)):
                checkpoint_names.add(key)
                if key not in model_names:
                    continue
                tensor = handle.get_tensor(key)
                if tensor.is_floating_point():
                    tensor = tensor.to(BACKBONE_DTYPE)
                set_module_tensor_to_device(
                    model, key, device_for_tensor(key, device_map), value=tensor, dtype=BACKBONE_DTYPE
                )
                del tensor
                loaded += 1
    return {
        'loaded': loaded,
        'missing': sorted(model_names - checkpoint_names),
        'unused_checkpoint_keys': sorted(checkpoint_names - model_names),
    }


release_cuda_cache()
with init_empty_weights(include_buffers=False):
    BACKBONE = timm.create_model(TIMM_MODEL_NAME, pretrained=False, num_classes=0, img_size=IMAGE_SIZE)
PLACEMENT_PLAN = build_fixed_device_plan(BACKBONE)
CHECKPOINT_FILES, CHECKPOINT_TOTAL_GIB = resolve_checkpoint_files(MODEL_SNAPSHOT_DIR)
PLACEMENT_PLAN['checkpoint_files'] = [path.name for path in CHECKPOINT_FILES]
PLACEMENT_PLAN['checkpoint_total_gib'] = CHECKPOINT_TOTAL_GIB
PLACEMENT_PLAN['memory_before_load'] = cuda_memory_snapshot('before_load')
write_json_atomic(PLACEMENT_PLAN, ARTIFACT_DIR / 'planned_device_map.json')

load_started = time.perf_counter()
try:
    LOAD_REPORT = stream_checkpoint(BACKBONE, CHECKPOINT_FILES, PLACEMENT_PLAN['device_map'])
    if LOAD_REPORT['missing']:
        raise RuntimeError('Checkpoint is missing model tensors: {}'.format(LOAD_REPORT['missing'][:10]))
    remaining_meta = [
        name for name, tensor in list(BACKBONE.named_parameters()) + list(BACKBONE.named_buffers())
        if tensor.device.type == 'meta'
    ]
    if remaining_meta:
        raise RuntimeError('Unmaterialized meta tensors remain: {}'.format(remaining_meta[:10]))
    BACKBONE = dispatch_model(BACKBONE, device_map=PLACEMENT_PLAN['device_map'], force_hooks=True)
except Exception as exc:
    write_json_atomic({
        'error': '{}: {}'.format(type(exc).__name__, exc),
        'plan': PLACEMENT_PLAN,
        'memory_after_failure': cuda_memory_snapshot('load_failure'),
        'action': 'Restart a clean Kaggle T4 x2 session. Feature batch size does not solve weight-loading OOM.',
    }, ARTIFACT_DIR / 'model_load_failure_report.json')
    raise
MODEL_LOAD_SECONDS = time.perf_counter() - load_started
BACKBONE.eval().requires_grad_(False)

ACTUAL_DEVICE_MAP = {
    name: normalized_device(device) for name, device in getattr(BACKBONE, 'hf_device_map', {}).items()
}
PLANNED_NORMALIZED_DEVICE_MAP = {
    name: normalized_device(device) for name, device in PLACEMENT_PLAN['device_map'].items()
}
if not ACTUAL_DEVICE_MAP or set(ACTUAL_DEVICE_MAP.values()) != {'cuda:0', 'cuda:1'}:
    raise RuntimeError('Accelerate did not attach an exact two-GPU map.')
if ACTUAL_DEVICE_MAP != PLANNED_NORMALIZED_DEVICE_MAP:
    raise RuntimeError('Accelerate actual device map differs from the persisted fixed plan.')
if any(device in {'cpu', 'disk', 'meta'} for device in ACTUAL_DEVICE_MAP.values()):
    raise RuntimeError('CPU/disk/meta offload is forbidden.')
BLOCK_DEVICES = []
for index, block in enumerate(BACKBONE.blocks):
    devices = {str(parameter.device) for parameter in block.parameters()}
    if len(devices) != 1:
        raise RuntimeError('Transformer block {} is split across devices.'.format(index))
    BLOCK_DEVICES.append(devices.pop())
EXPECTED_BLOCK_DEVICES = ['cuda:0'] * 20 + ['cuda:1'] * 20
if BLOCK_DEVICES != EXPECTED_BLOCK_DEVICES:
    raise RuntimeError('Actual block placement differs from the locked 0–19 / 20–39 split.')
if {str(parameter.device) for parameter in BACKBONE.parameters()} != {'cuda:0', 'cuda:1'}:
    raise RuntimeError('Backbone parameters are not placed exclusively across both CUDA devices.')
if {parameter.dtype for parameter in BACKBONE.parameters() if parameter.is_floating_point()} != {torch.float16}:
    raise RuntimeError('Backbone floating weights are not uniformly FP16.')
if any(parameter.requires_grad for parameter in BACKBONE.parameters()):
    raise RuntimeError('Backbone is not fully frozen.')
for module_name, module in BACKBONE.named_modules():
    for buffer_name, buffer in module._buffers.items():
        if buffer is None or buffer_name in module._non_persistent_buffers_set:
            continue
        if str(buffer.device) not in {'cuda:0', 'cuda:1'}:
            raise RuntimeError('Persistent buffer {}.{} remains on {}.'.format(
                module_name, buffer_name, buffer.device
            ))

NUM_PREFIX_TOKENS = int(BACKBONE.num_prefix_tokens)
NUM_REGISTER_TOKENS = int(BACKBONE.reg_token.shape[1])
HIDDEN_SIZE = int(BACKBONE.num_features)
PATCH_INPUT_DEVICE = next(BACKBONE.patch_embed.proj.parameters()).device
FINAL_NORM_DEVICE = next(BACKBONE.norm.parameters()).device
if (
    len(BACKBONE.blocks) != EXPECTED_BLOCKS
    or NUM_PREFIX_TOKENS != EXPECTED_PREFIX_TOKENS
    or NUM_REGISTER_TOKENS != EXPECTED_REGISTER_TOKENS
    or HIDDEN_SIZE != EXPECTED_HIDDEN_SIZE
    or str(PATCH_INPUT_DEVICE) != 'cuda:0'
    or str(FINAL_NORM_DEVICE) != 'cuda:1'
):
    raise RuntimeError('Runtime DINOv3 model contract differs from the locked V11 contract.')
resolved_data = timm.data.resolve_model_data_config(BACKBONE)
if not np.allclose(resolved_data['mean'], IMAGENET_MEAN) or not np.allclose(resolved_data['std'], IMAGENET_STD):
    raise RuntimeError('Resolved normalization differs from the locked V10 preprocessing constants.')
POST_LOAD_MEMORY = cuda_memory_snapshot('after_load')
if any(row['free_gib'] < GPU_MIN_HEADROOM_GIB for row in POST_LOAD_MEMORY):
    raise RuntimeError('Actual post-load CUDA headroom is below {:.1f} GiB; restart a clean T4 x2 session.'.format(
        GPU_MIN_HEADROOM_GIB
    ))

MODEL_MANIFEST = {
    'repository': MODEL_REPO_ID,
    'immutable_revision': RESOLVED_REVISION,
    'snapshot_files': [path.name for path in CHECKPOINT_FILES],
    'checkpoint_total_gib': CHECKPOINT_TOTAL_GIB,
    'download_seconds': MODEL_DOWNLOAD_SECONDS,
    'load_seconds': MODEL_LOAD_SECONDS,
    'model_class': type(BACKBONE).__name__,
    'parameter_count': sum(parameter.numel() for parameter in BACKBONE.parameters()),
    'trainable_parameter_count': 0,
    'blocks': len(BACKBONE.blocks),
    'hidden_size': HIDDEN_SIZE,
    'prefix_tokens': NUM_PREFIX_TOKENS,
    'register_tokens': NUM_REGISTER_TOKENS,
    'patch_tokens': EXPECTED_PATCH_TOKENS,
    'register_tokens_excluded': True,
    'final_norm_applied_times': 1,
    'backbone_dtype': 'float16',
    'placement': PLACEMENT_PLAN,
    'load_report': LOAD_REPORT,
}
write_json_atomic(MODEL_MANIFEST, ARTIFACT_DIR / 'model_manifest.json')
write_json_atomic({
    'actual_device_map': ACTUAL_DEVICE_MAP,
    'block_devices': BLOCK_DEVICES,
    'memory_after_load': POST_LOAD_MEMORY,
}, ARTIFACT_DIR / 'device_map.json')
print('Model loaded:', MODEL_MANIFEST['parameter_count'], 'parameters')
print('Blocks:', BLOCK_DEVICES.count('cuda:0'), 'on cuda:0 +', BLOCK_DEVICES.count('cuda:1'), 'on cuda:1')


## 6. Persisted JL projection and Phase 0 preflight

Preflight separates semantic equivalence (same forward, FP32 reductions) from the intentional FP16 cache-quantization check. It also tests view determinism, patch order/shape, projection norm distortion on seeded probes, finite values, cache budget, explicit placement, and measured throughput before full extraction.

In [ ]:
def projection_checksum(matrix):
    array = matrix.detach().cpu().contiguous().numpy()
    return hashlib.sha256(array.tobytes()).hexdigest()


def build_or_load_projection():
    expected_metadata = {
        'kind': 'seeded Rademacher Johnson-Lindenstrauss projection',
        'seed': PROJECTION_SEED,
        'input_dim': EXPECTED_HIDDEN_SIZE,
        'output_dim': PROJECTION_DIM,
        'scale': '1/sqrt(output_dim)',
    }
    if PROJECTION_STATE_PATH.is_file():
        payload = torch.load(PROJECTION_STATE_PATH, map_location='cpu', weights_only=False)
        if payload.get('metadata') != expected_metadata:
            raise RuntimeError('Existing projection metadata differs from the locked V11 recipe.')
        matrix = payload['matrix'].float().contiguous()
    else:
        generator = torch.Generator(device='cpu').manual_seed(PROJECTION_SEED)
        matrix = torch.randint(
            0, 2, (EXPECTED_HIDDEN_SIZE, PROJECTION_DIM), generator=generator, dtype=torch.int8
        ).float()
        matrix.mul_(2).sub_(1).div_(math.sqrt(PROJECTION_DIM))
        atomic_torch_save({'metadata': expected_metadata, 'matrix': matrix}, PROJECTION_STATE_PATH)
    if tuple(matrix.shape) != (EXPECTED_HIDDEN_SIZE, PROJECTION_DIM) or not torch.isfinite(matrix).all():
        raise RuntimeError('Persisted projection has invalid shape or values.')
    checksum = projection_checksum(matrix)
    manifest = {
        **expected_metadata,
        'state_file': PROJECTION_STATE_PATH.name,
        'state_file_sha256': sha256_file(PROJECTION_STATE_PATH),
        'matrix_sha256': checksum,
        'matrix_shape': list(matrix.shape),
        'matrix_dtype': str(matrix.dtype),
        'train_fitted': False,
        'fold_independent': True,
    }
    write_json_atomic(manifest, ARTIFACT_DIR / 'projection_manifest.json')
    return matrix, checksum, manifest


PROJECTION_CPU, PROJECTION_CHECKSUM, PROJECTION_MANIFEST = build_or_load_projection()
PROJECTION_GPU = PROJECTION_CPU.to(FINAL_NORM_DEVICE, non_blocking=True)


def identify_prefix_and_patches(item, batch_size):
    if not isinstance(item, (tuple, list)) or len(item) != 2:
        raise RuntimeError('Expected timm (spatial, prefix) token pair.')
    first, second = item
    if first.ndim == 3 and first.shape[1] == EXPECTED_PREFIX_TOKENS:
        prefix, patches = first, second
    elif second.ndim == 3 and second.shape[1] == EXPECTED_PREFIX_TOKENS:
        prefix, patches = second, first
    else:
        raise RuntimeError('Cannot identify prefix tokens from {} and {}.'.format(first.shape, second.shape))
    if tuple(prefix.shape) != (batch_size, EXPECTED_PREFIX_TOKENS, EXPECTED_HIDDEN_SIZE):
        raise RuntimeError('Prefix token shape contract failed: {}.'.format(tuple(prefix.shape)))
    if tuple(patches.shape) != (batch_size, EXPECTED_PATCH_TOKENS, EXPECTED_HIDDEN_SIZE):
        raise RuntimeError('True-patch shape contract failed: {}.'.format(tuple(patches.shape)))
    if prefix.dtype != torch.float16 or patches.dtype != torch.float16:
        raise RuntimeError('Normalized backbone tokens must be FP16.')
    return prefix, patches


def normalized_final_tokens(model_input):
    outputs = BACKBONE.forward_intermediates(
        model_input,
        indices=[EXPECTED_BLOCKS - 1],
        return_prefix_tokens=True,
        norm=True,
        output_fmt='NLC',
        intermediates_only=True,
    )
    if not isinstance(outputs, (tuple, list)) or len(outputs) != 1:
        raise RuntimeError('Expected exactly one final normalized intermediate.')
    return identify_prefix_and_patches(outputs[0], model_input.shape[0])


def representations_from_tokens(prefix, patches):
    cls = prefix[:, 0].float()
    true_patch_mean = patches.float().mean(dim=1)
    control = torch.cat((cls, true_patch_mean), dim=1)
    patch_grid = patches.float().reshape(-1, PATCH_GRID, PATCH_GRID, EXPECTED_HIDDEN_SIZE)
    pooled = patch_grid.reshape(
        -1, POOLED_GRID, 2, POOLED_GRID, 2, EXPECTED_HIDDEN_SIZE
    ).mean(dim=(2, 4)).reshape(-1, SPATIAL_TOKENS, EXPECTED_HIDDEN_SIZE)
    projected_spatial = pooled @ PROJECTION_GPU
    projected_cls = cls @ PROJECTION_GPU
    projected_mean = true_patch_mean @ PROJECTION_GPU
    candidate = torch.cat((
        projected_spatial,
        projected_cls.unsqueeze(1),
        projected_mean.unsqueeze(1),
    ), dim=1)
    if tuple(control.shape[1:]) != (CONTROL_DIM,) or tuple(candidate.shape[1:]) != (CANDIDATE_TOKENS, PROJECTION_DIM):
        raise RuntimeError('Compressed representation shape contract failed.')
    if not torch.isfinite(control).all() or not torch.isfinite(candidate).all():
        raise RuntimeError('Compressed representation contains NaN or Inf.')
    return control, candidate


def extract_representations(model_input):
    with torch.inference_mode():
        prefix, patches = normalized_final_tokens(model_input.to(PATCH_INPUT_DEVICE, dtype=BACKBONE_DTYPE))
        return representations_from_tokens(prefix, patches)


sample_rows = train_df.iloc[:FEATURE_BATCH_SIZE]
sample_images = [load_rgb_image(path) for path in sample_rows['path']]
original_batch = torch.stack([
    make_view(image, image_id, 'original')
    for image, image_id in zip(sample_images, sample_rows['image_id'])
])
view_checks = {}
for image, image_id in zip(sample_images, sample_rows['image_id']):
    original = make_view(image, image_id, 'original')
    for view_name in VIEWS[1:]:
        first = make_view(image, image_id, view_name)
        second = make_view(image, image_id, view_name)
        view_checks.setdefault(view_name, []).append({
            'bitwise_reproducible': bool(torch.equal(first, second)),
            'mean_abs_change_from_original': float((first - original).abs().mean()),
        })
if not all(row['bitwise_reproducible'] for rows in view_checks.values() for row in rows):
    raise RuntimeError('A seeded train view is not reproducible.')
if not all(row['mean_abs_change_from_original'] > 1e-4 for rows in view_checks.values() for row in rows):
    raise RuntimeError('A material view is unexpectedly identical to the original.')

with torch.inference_mode():
    probe_input = original_batch.to(PATCH_INPUT_DEVICE, dtype=BACKBONE_DTYPE)
    prefix, patches = normalized_final_tokens(probe_input)
    manual_final = torch.cat((prefix, patches), dim=1).float()
    official_final = BACKBONE.forward_features(probe_input).to(manual_final.device).float()
    norm_difference = (manual_final - official_final).abs()
    final_norm_pass = bool(torch.allclose(manual_final, official_final, atol=2e-3, rtol=2e-3))
    control_fp32, candidate_fp32 = representations_from_tokens(prefix, patches)
    control_reference = torch.cat((prefix[:, 0].float(), patches.float().mean(dim=1)), dim=1)
    control_semantic_max_abs = float((control_fp32 - control_reference).abs().max())
if not final_norm_pass:
    raise RuntimeError('Final normalized intermediate differs from timm forward_features.')
if control_semantic_max_abs > 1e-7:
    raise RuntimeError('Control representation differs from explicit CLS + true-patch mean.')

with torch.inference_mode():
    nchw_outputs = BACKBONE.forward_intermediates(
        probe_input,
        indices=[EXPECTED_BLOCKS - 1],
        return_prefix_tokens=True,
        norm=True,
        output_fmt='NCHW',
        intermediates_only=True,
    )
nchw_pair = nchw_outputs[0]
nchw_spatial = next(tensor for tensor in nchw_pair if tensor.ndim == 4)
nchw_as_nlc = nchw_spatial.permute(0, 2, 3, 1).reshape_as(patches).float()
patch_order_max_abs = float((nchw_as_nlc - patches.float()).abs().max())
if not torch.allclose(nchw_as_nlc, patches.float(), atol=2e-3, rtol=2e-3):
    raise RuntimeError('NLC patch order does not match timm NCHW spatial order.')

synthetic_grid = torch.arange(256, dtype=torch.float32).reshape(1, 16, 16, 1)
synthetic_pool = synthetic_grid.reshape(1, 8, 2, 8, 2, 1).mean(dim=(2, 4)).reshape(8, 8)
if synthetic_pool[0, 0].item() != 8.5 or synthetic_pool[-1, -1].item() != 246.5:
    raise RuntimeError('2x2 row-major pooling self-check failed.')

projection_generator = torch.Generator().manual_seed(PROJECTION_SEED + 1)
projection_probes = torch.randn(64, EXPECTED_HIDDEN_SIZE, generator=projection_generator)
projected_probes = projection_probes @ PROJECTION_CPU
norm_ratios = projected_probes.norm(dim=1) / projection_probes.norm(dim=1)
projection_max_distortion = float((norm_ratios - 1.0).abs().max())
if projection_max_distortion > 0.30 or not torch.isfinite(projected_probes).all():
    raise RuntimeError('JL projection sanity check failed with max norm distortion {:.3f}.'.format(
        projection_max_distortion
    ))

control_quantized = control_fp32.half().float()
control_fp16_max_abs = float((control_fp32 - control_quantized).abs().max())
if control_fp16_max_abs > 0.02:
    raise RuntimeError('Control FP16 cache quantization error is unexpectedly large.')
candidate_quantized = candidate_fp32.half().float()
candidate_fp16_max_abs = float((candidate_fp32 - candidate_quantized).abs().max())
if not torch.isfinite(candidate_quantized).all():
    raise RuntimeError('Projected candidate values overflow FP16 cache storage.')

candidate_view_bytes = EXPECTED_TRAIN_ROWS * CANDIDATE_TOKENS * PROJECTION_DIM * 2
control_bytes = EXPECTED_TRAIN_ROWS * CONTROL_DIM * 2
train_cache_bytes = len(VIEWS) * candidate_view_bytes + control_bytes
TRAIN_CACHE_ESTIMATE_GIB = train_cache_bytes / 2 ** 30
test_cache_max_bytes = EXPECTED_TEST_ROWS * max(CONTROL_DIM, CANDIDATE_TOKENS * PROJECTION_DIM) * 2
TEST_CACHE_MAX_ESTIMATE_GIB = test_cache_max_bytes / 2 ** 30
TOTAL_FEATURE_CACHE_ESTIMATE_GIB = TRAIN_CACHE_ESTIMATE_GIB + TEST_CACHE_MAX_ESTIMATE_GIB
HEAD_INPUT_BATCH_ESTIMATE_MIB = (
    HEAD_BATCH_SIZE * len(VIEWS) * CANDIDATE_TOKENS * PROJECTION_DIM * 4 / 2 ** 20
)
if TOTAL_FEATURE_CACHE_ESTIMATE_GIB >= TRAIN_CACHE_BUDGET_GIB:
    raise RuntimeError('Projected feature caches {:.2f} GiB exceed the locked budget.'.format(
        TOTAL_FEATURE_CACHE_ESTIMATE_GIB
    ))
working_disk_free_gib = shutil.disk_usage(KAGGLE_WORKING).free / 2 ** 30
working_disk_margin_after_feature_caches_gib = working_disk_free_gib - TOTAL_FEATURE_CACHE_ESTIMATE_GIB
if working_disk_margin_after_feature_caches_gib < MIN_WORKING_DISK_MARGIN_GIB:
    raise RuntimeError('Only {:.2f} GiB would remain after projected feature caches; {:.2f} GiB is required.'.format(
        working_disk_margin_after_feature_caches_gib, MIN_WORKING_DISK_MARGIN_GIB
    ))

for _ in range(2):
    extract_representations(original_batch)
torch.cuda.synchronize(0)
torch.cuda.synchronize(1)
benchmark_started = time.perf_counter()
benchmark_repeats = 6
for _ in range(benchmark_repeats):
    extract_representations(original_batch)
torch.cuda.synchronize(0)
torch.cuda.synchronize(1)
benchmark_seconds = time.perf_counter() - benchmark_started
throughput = benchmark_repeats * len(original_batch) / benchmark_seconds
if throughput < MIN_EXTRACTION_IMAGES_PER_SECOND:
    raise RuntimeError('Extraction throughput {:.2f} images/s is below {:.2f}.'.format(
        throughput, MIN_EXTRACTION_IMAGES_PER_SECOND
    ))
estimated_train_forward_hours = EXPECTED_TRAIN_ROWS * len(VIEWS) / throughput / 3600.0
estimated_test_forward_minutes = EXPECTED_TEST_ROWS / throughput / 60.0

PREFLIGHT_REPORT = {
    'passed': True,
    'immutable_model_revision': RESOLVED_REVISION,
    'checkpoint_files': [path.name for path in CHECKPOINT_FILES],
    'exact_two_t4': GPU_RUNTIME_INFO,
    'planned_block_split': {'cuda:0': [0, 19], 'cuda:1': [20, 39]},
    'actual_block_devices': BLOCK_DEVICES,
    'token_shape': list(manual_final.shape),
    'prefix_tokens': EXPECTED_PREFIX_TOKENS,
    'register_tokens_excluded': EXPECTED_REGISTER_TOKENS,
    'patch_grid': [PATCH_GRID, PATCH_GRID],
    'hidden_size': EXPECTED_HIDDEN_SIZE,
    'final_norm': {
        'applied_times': 1,
        'reference': 'timm forward_features',
        'max_abs_difference': float(norm_difference.max()),
        'mean_abs_difference': float(norm_difference.mean()),
        'atol': 0.002,
        'rtol': 0.002,
    },
    'patch_order_nlc_vs_nchw_max_abs': patch_order_max_abs,
    'pooling_shape': [POOLED_GRID, POOLED_GRID],
    'projection_shape': list(PROJECTION_CPU.shape),
    'projection_checksum': PROJECTION_CHECKSUM,
    'projection_max_probe_norm_distortion': projection_max_distortion,
    'candidate_feature_shape': list(candidate_fp32.shape),
    'control_feature_shape': list(control_fp32.shape),
    'control_semantic_reference_max_abs': control_semantic_max_abs,
    'features_finite': bool(torch.isfinite(candidate_fp32).all() and torch.isfinite(control_fp32).all()),
    'control_fp16_quantization_max_abs': control_fp16_max_abs,
    'candidate_fp16_quantization_max_abs': candidate_fp16_max_abs,
    'view_checks': view_checks,
    'train_cache_estimate_gib': TRAIN_CACHE_ESTIMATE_GIB,
    'maximum_selected_test_cache_estimate_gib': TEST_CACHE_MAX_ESTIMATE_GIB,
    'total_feature_cache_estimate_gib': TOTAL_FEATURE_CACHE_ESTIMATE_GIB,
    'cache_budget_gib': TRAIN_CACHE_BUDGET_GIB,
    'working_disk_free_before_cache_gib': working_disk_free_gib,
    'working_disk_margin_after_feature_caches_gib': working_disk_margin_after_feature_caches_gib,
    'minimum_working_disk_margin_gib': MIN_WORKING_DISK_MARGIN_GIB,
    'candidate_head_input_batch_mib': HEAD_INPUT_BATCH_ESTIMATE_MIB,
    'throughput_images_per_second': throughput,
    'minimum_throughput_images_per_second': MIN_EXTRACTION_IMAGES_PER_SECOND,
    'estimated_three_view_train_forward_hours': estimated_train_forward_hours,
    'estimated_original_view_test_forward_minutes': estimated_test_forward_minutes,
    'test_images_accessed': False,
}
write_json_atomic(PREFLIGHT_REPORT, PREFLIGHT_REPORT_PATH)
del sample_images, original_batch, probe_input, prefix, patches, manual_final, official_final
del control_fp32, candidate_fp32, control_reference, control_quantized, candidate_quantized
del nchw_outputs, nchw_pair, nchw_spatial, nchw_as_nlc
release_cuda_cache()
print('Preflight passed | feature caches <= {:.2f} GiB | throughput {:.2f} images/s'.format(
    TOTAL_FEATURE_CACHE_ESTIMATE_GIB, throughput
))


## 7. Resumable, signed train cache

Every shard is atomic and verified before reuse. Its signature binds ordered source identities, labels/folds, immutable model revision, preprocessing and view recipes, projection checksum, tensor layout, and storage dtype. Test paths are not discovered here.

In [ ]:
TRAIN_CACHE_IDENTITY = {
    'schema_version': 1,
    'revision_marker': REVISION_MARKER,
    'model_revision': RESOLVED_REVISION,
    'data_contract_signature': DATA_CONTRACT_SIGNATURE,
    'fold_assignment_signature': FOLD_ASSIGNMENT_SIGNATURE,
    'ordered_image_ids_signature': ORDERED_IMAGE_IDS_SIGNATURE,
    'source_identity_signature': SOURCE_IDENTITY_SIGNATURE,
    'preprocess_signature': PREPROCESS_SIGNATURE,
    'view_signature': VIEW_SIGNATURE,
    'views': list(VIEWS),
    'projection_checksum': PROJECTION_CHECKSUM,
    'candidate_layout': ['spatial_8x8_row_major', 'projected_cls', 'projected_true_patch_mean'],
    'candidate_shape_per_row': [CANDIDATE_TOKENS, PROJECTION_DIM],
    'control_shape_per_row': [CONTROL_DIM],
    'storage_dtype': 'float16',
    'shard_size': FEATURE_SHARD_SIZE,
    'row_order': 'train_df row order',
}
TRAIN_CACHE_SIGNATURE = sha256_json(TRAIN_CACHE_IDENTITY)


class TrainViewRows(Dataset):
    def __init__(self, frame):
        self.paths = frame['path'].tolist()
        self.image_ids = frame['image_id'].tolist()

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, index):
        image = load_rgb_image(self.paths[index])
        image_id = self.image_ids[index]
        return {view_name: make_view(image, image_id, view_name) for view_name in VIEWS}


def expected_shard_ranges(row_count, shard_size):
    return [(start, min(start + shard_size, row_count)) for start in range(0, row_count, shard_size)]


def shard_path(start, stop):
    return TRAIN_CACHE_DIR / 'train_{:06d}_{:06d}.pt'.format(start, stop)


def validate_train_shard(path, start, stop):
    payload = torch.load(path, map_location='cpu', weights_only=False)
    metadata = payload.get('metadata', {})
    expected_ids = train_df.iloc[start:stop]['image_id'].tolist()
    checks = [
        metadata.get('cache_signature') == TRAIN_CACHE_SIGNATURE,
        metadata.get('start') == start,
        metadata.get('stop') == stop,
        metadata.get('image_ids_signature') == sha256_json(expected_ids),
        tuple(payload['row_indices'].shape) == (stop - start,),
        tuple(payload['control_original'].shape) == (stop - start, CONTROL_DIM),
        payload['control_original'].dtype == torch.float16,
    ]
    for view_name in VIEWS:
        tensor = payload['candidate_' + view_name]
        checks.extend([
            tuple(tensor.shape) == (stop - start, CANDIDATE_TOKENS, PROJECTION_DIM),
            tensor.dtype == torch.float16,
            bool(torch.isfinite(tensor).all()),
        ])
    checks.append(bool(torch.isfinite(payload['control_original']).all()))
    expected_indices = torch.arange(start, stop, dtype=torch.int64)
    checks.append(bool(torch.equal(payload['row_indices'], expected_indices)))
    if not all(checks):
        raise RuntimeError('Existing cache shard failed validation: {}. Remove only this V11 cache and rerun.'.format(path))
    return payload


train_ranges = expected_shard_ranges(len(train_df), FEATURE_SHARD_SIZE)
train_dataset = TrainViewRows(train_df)
for start, stop in tqdm(train_ranges, desc='train feature shards'):
    path = shard_path(start, stop)
    if path.is_file():
        validate_train_shard(path, start, stop)
        continue
    subset = torch.utils.data.Subset(train_dataset, range(start, stop))
    loader = DataLoader(
        subset,
        batch_size=FEATURE_BATCH_SIZE,
        shuffle=False,
        num_workers=FEATURE_LOADER_WORKERS,
        pin_memory=True,
        persistent_workers=FEATURE_LOADER_WORKERS > 0,
    )
    control_parts = []
    candidate_parts = {view_name: [] for view_name in VIEWS}
    for batch in tqdm(loader, leave=False, desc='rows {}:{}'.format(start, stop)):
        control, candidate = extract_representations(batch['original'])
        control_parts.append(control.half().cpu())
        candidate_parts['original'].append(candidate.half().cpu())
        del control, candidate
        for view_name in VIEWS[1:]:
            unused_control, candidate = extract_representations(batch[view_name])
            candidate_parts[view_name].append(candidate.half().cpu())
            del unused_control, candidate
    payload = {
        'metadata': {
            'cache_signature': TRAIN_CACHE_SIGNATURE,
            'start': start,
            'stop': stop,
            'image_ids_signature': sha256_json(train_df.iloc[start:stop]['image_id'].tolist()),
        },
        'row_indices': torch.arange(start, stop, dtype=torch.int64),
        'labels': torch.tensor(train_df.iloc[start:stop]['label'].to_numpy(), dtype=torch.int64),
        'folds': torch.tensor(train_df.iloc[start:stop]['fold'].to_numpy(), dtype=torch.int64),
        'control_original': torch.cat(control_parts),
    }
    for view_name in VIEWS:
        payload['candidate_' + view_name] = torch.cat(candidate_parts[view_name])
    atomic_torch_save(payload, path)
    validate_train_shard(path, start, stop)
    del payload, loader, subset, control_parts, candidate_parts
    release_cuda_cache()

cached_control = torch.empty((len(train_df), CONTROL_DIM), dtype=torch.float16)
cached_candidate = {
    view_name: torch.empty((len(train_df), CANDIDATE_TOKENS, PROJECTION_DIM), dtype=torch.float16)
    for view_name in VIEWS
}
cached_labels = torch.empty(len(train_df), dtype=torch.int64)
cached_folds = torch.empty(len(train_df), dtype=torch.int64)
SHARD_MANIFEST_ROWS = []
for start, stop in tqdm(train_ranges, desc='validating and joining cache'):
    path = shard_path(start, stop)
    payload = validate_train_shard(path, start, stop)
    cached_control[start:stop].copy_(payload['control_original'])
    for view_name in VIEWS:
        cached_candidate[view_name][start:stop].copy_(payload['candidate_' + view_name])
    cached_labels[start:stop].copy_(payload['labels'])
    cached_folds[start:stop].copy_(payload['folds'])
    SHARD_MANIFEST_ROWS.append({
        'file': path.name,
        'start': start,
        'stop': stop,
        'bytes': path.stat().st_size,
        'sha256': sha256_file(path),
    })
    del payload

if not torch.equal(cached_labels, torch.tensor(train_df['label'].to_numpy(), dtype=torch.int64)):
    raise RuntimeError('Cached labels differ from canonical row order.')
if not torch.equal(cached_folds, torch.tensor(train_df['fold'].to_numpy(), dtype=torch.int64)):
    raise RuntimeError('Cached folds differ from canonical row order.')
if not torch.isfinite(cached_control).all() or not all(torch.isfinite(value).all() for value in cached_candidate.values()):
    raise RuntimeError('Joined train cache contains NaN or Inf.')

TRAIN_CACHE_MANIFEST = {
    'cache_identity': TRAIN_CACHE_IDENTITY,
    'cache_signature': TRAIN_CACHE_SIGNATURE,
    'complete': True,
    'rows': len(train_df),
    'shards': SHARD_MANIFEST_ROWS,
    'total_shard_bytes': sum(row['bytes'] for row in SHARD_MANIFEST_ROWS),
    'estimated_gib_before_extraction': TRAIN_CACHE_ESTIMATE_GIB,
    'control_shape': list(cached_control.shape),
    'candidate_shapes': {name: list(value.shape) for name, value in cached_candidate.items()},
    'storage_dtype': 'float16',
    'test_images_accessed': False,
}
write_json_atomic(TRAIN_CACHE_MANIFEST, ARTIFACT_DIR / 'cache_manifest_train.json')
print('Train cache ready:', TRAIN_CACHE_SIGNATURE)


## 8. Two heads, one training/evaluation path, and locked metrics

`timm_control` is the exact V10-style LayerNorm→512 MLP over final CLS + true-patch mean. `material_query_attention` is the only V11 candidate. Both use the same folds, seed, optimizer, class weighting, label smoothing, scheduler, clipping, early stopping, and metric code. The recomputed fold assignment must equal persisted V10 SHA-256 `0a06976e…305c` before training can begin. Candidate validation always uses its original-view representation.

In [ ]:
HEAD_DEVICE = torch.device('cuda:0')


class TimmControlHead(nn.Module):
    def __init__(self):
        super().__init__()
        self.network = nn.Sequential(
            nn.LayerNorm(CONTROL_DIM),
            nn.Linear(CONTROL_DIM, CONTROL_HIDDEN_DIM),
            nn.GELU(),
            nn.Dropout(HEAD_DROPOUT),
            nn.Linear(CONTROL_HIDDEN_DIM, NUM_CLASSES),
        )

    def forward(self, features, return_attention=False):
        logits = self.network(features)
        return (logits, None) if return_attention else logits


class MaterialQueryAttentionHead(nn.Module):
    def __init__(self):
        super().__init__()
        self.spatial_norm = nn.LayerNorm(PROJECTION_DIM)
        self.global_norm = nn.LayerNorm(PROJECTION_DIM)
        self.position = nn.Parameter(torch.empty(POOLED_GRID, POOLED_GRID, PROJECTION_DIM))
        self.queries = nn.Parameter(torch.empty(MATERIAL_QUERIES, PROJECTION_DIM))
        self.cross_attention = nn.MultiheadAttention(
            PROJECTION_DIM, ATTENTION_HEADS, dropout=0.10, batch_first=True
        )
        self.query_norm = nn.LayerNorm(PROJECTION_DIM)
        self.ffn_norm = nn.LayerNorm(PROJECTION_DIM)
        self.ffn = nn.Sequential(
            nn.Linear(PROJECTION_DIM, MATERIAL_FFN_DIM),
            nn.GELU(),
            nn.Dropout(0.10),
            nn.Linear(MATERIAL_FFN_DIM, PROJECTION_DIM),
        )
        fused_dim = (MATERIAL_QUERIES + GLOBAL_TOKENS) * PROJECTION_DIM
        self.classifier = nn.Sequential(
            nn.LayerNorm(fused_dim),
            nn.Linear(fused_dim, CONTROL_HIDDEN_DIM),
            nn.GELU(),
            nn.Dropout(HEAD_DROPOUT),
            nn.Linear(CONTROL_HIDDEN_DIM, NUM_CLASSES),
        )
        nn.init.trunc_normal_(self.position, std=0.02)
        nn.init.trunc_normal_(self.queries, std=0.02)

    def forward(self, features, return_attention=False):
        if tuple(features.shape[1:]) != (CANDIDATE_TOKENS, PROJECTION_DIM):
            raise RuntimeError('Material head received shape {}.'.format(tuple(features.shape)))
        spatial = self.spatial_norm(features[:, :SPATIAL_TOKENS]) + self.position.flatten(0, 1).unsqueeze(0)
        globals_ = self.global_norm(features[:, SPATIAL_TOKENS:])
        queries = self.queries.unsqueeze(0).expand(features.shape[0], -1, -1)
        attended, weights = self.cross_attention(
            queries,
            spatial,
            spatial,
            need_weights=return_attention,
            average_attn_weights=False,
        )
        queries = self.query_norm(queries + attended)
        queries = self.ffn_norm(queries + self.ffn(queries))
        logits = self.classifier(torch.cat((queries.flatten(1), globals_.flatten(1)), dim=1))
        return (logits, weights) if return_attention else logits


def build_head(candidate_name):
    if candidate_name == CONTROL_NAME:
        return TimmControlHead()
    if candidate_name == CANDIDATE_NAME:
        return MaterialQueryAttentionHead()
    raise KeyError('Unknown head: {}'.format(candidate_name))


HEAD_SPECS = {
    CONTROL_NAME: {
        'representation': 'final normalized CLS + true-patch mean, registers excluded',
        'architecture': 'LayerNorm(8192) -> Linear(512) -> GELU -> Dropout(0.4) -> Linear(3)',
        'train_views': ['original'],
        'validation_view': 'original',
        'projection_checksum': None,
    },
    CANDIDATE_NAME: {
        'representation': '8x8 projected final patches + projected CLS + projected true-patch mean',
        'architecture': 'four learned queries, one 8-head cross-attention block, one FFN, 512-D classifier',
        'train_views': list(VIEWS),
        'validation_view': 'original',
        'projection_checksum': PROJECTION_CHECKSUM,
    },
}
for candidate_name in (CONTROL_NAME, CANDIDATE_NAME):
    probe = build_head(candidate_name)
    HEAD_SPECS[candidate_name]['parameter_count'] = sum(parameter.numel() for parameter in probe.parameters())
    with torch.inference_mode():
        shape = (2, CONTROL_DIM) if candidate_name == CONTROL_NAME else (2, CANDIDATE_TOKENS, PROJECTION_DIM)
        probe_output = probe(torch.zeros(shape))
    if tuple(probe_output.shape) != (2, NUM_CLASSES):
        raise RuntimeError('Head shape self-check failed for {}.'.format(candidate_name))
    del probe, probe_output


class CachedHeadRows(Dataset):
    def __init__(self, candidate_name, indices, include_labels=True):
        self.candidate_name = candidate_name
        self.indices = [int(index) for index in indices]
        self.include_labels = include_labels

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, position):
        index = self.indices[position]
        if self.candidate_name == CONTROL_NAME:
            features = cached_control[index]
        else:
            features = {view_name: cached_candidate[view_name][index] for view_name in VIEWS}
        label = cached_labels[index] if self.include_labels else torch.tensor(-1, dtype=torch.int64)
        return features, label, index


def class_weights_for(labels):
    counts = np.bincount(np.asarray(labels, dtype=np.int64), minlength=NUM_CLASSES)
    if (counts == 0).any():
        raise RuntimeError('Training partition is missing a class.')
    weights = counts.sum() / (NUM_CLASSES * counts)
    return counts, torch.tensor(weights, dtype=torch.float32, device=HEAD_DEVICE)


def symmetric_js(logits_a, logits_b):
    probability_a = logits_a.softmax(dim=1).clamp_min(1e-7)
    probability_b = logits_b.softmax(dim=1).clamp_min(1e-7)
    midpoint = (0.5 * (probability_a + probability_b)).clamp_min(1e-7)
    return 0.5 * (
        F.kl_div(midpoint.log(), probability_a, reduction='batchmean')
        + F.kl_div(midpoint.log(), probability_b, reduction='batchmean')
    )


def batch_loss(model, candidate_name, features, labels, criterion):
    if candidate_name == CONTROL_NAME:
        logits = model(features.to(HEAD_DEVICE, dtype=torch.float32, non_blocking=True))
        return criterion(logits, labels)
    logits = {
        view_name: model(features[view_name].to(HEAD_DEVICE, dtype=torch.float32, non_blocking=True))
        for view_name in VIEWS
    }
    loss = sum(VIEW_CE_WEIGHTS[view_name] * criterion(logits[view_name], labels) for view_name in VIEWS)
    consistency = 0.5 * (
        symmetric_js(logits['original'], logits['color_debiased'])
        + symmetric_js(logits['original'], logits['local_material'])
    )
    return loss + CONSISTENCY_WEIGHT * consistency


def predict_probabilities(model, candidate_name, indices):
    expected_indices = [int(index) for index in indices]
    loader = DataLoader(
        CachedHeadRows(candidate_name, expected_indices),
        batch_size=PREDICT_BATCH_SIZE,
        shuffle=False,
        num_workers=0,
        pin_memory=True,
    )
    probabilities = []
    observed_indices = []
    model.eval()
    with torch.inference_mode():
        for features, _labels, batch_indices in loader:
            if candidate_name == CANDIDATE_NAME:
                features = features['original']
            logits = model(features.to(HEAD_DEVICE, dtype=torch.float32, non_blocking=True))
            batch_probabilities = logits.softmax(dim=1)
            if not torch.isfinite(batch_probabilities).all():
                raise RuntimeError('Prediction probabilities contain NaN or Inf.')
            probabilities.append(batch_probabilities.cpu())
            observed_indices.extend(int(index) for index in batch_indices.tolist())
    if observed_indices != expected_indices:
        raise RuntimeError('Prediction row order changed inside DataLoader.')
    result = torch.cat(probabilities).numpy().astype(np.float64)
    if result.shape != (len(expected_indices), NUM_CLASSES):
        raise RuntimeError('Probability matrix shape is invalid: {}.'.format(result.shape))
    return result


def metric_bundle(labels, probabilities):
    labels = np.asarray(labels, dtype=np.int64)
    probabilities = np.asarray(probabilities, dtype=np.float64)
    if probabilities.shape != (len(labels), NUM_CLASSES) or not np.isfinite(probabilities).all():
        raise RuntimeError('Metric inputs are incomplete or non-finite.')
    predictions = probabilities.argmax(axis=1)
    matrix = confusion_matrix(labels, predictions, labels=list(range(NUM_CLASSES)))
    report = classification_report(
        labels,
        predictions,
        labels=list(range(NUM_CLASSES)),
        target_names=list(CLASS_NAMES),
        output_dict=True,
        zero_division=0,
    )
    return {
        'rows': len(labels),
        'macro_f1': float(f1_score(labels, predictions, labels=list(range(NUM_CLASSES)), average='macro')),
        'accuracy': float(accuracy_score(labels, predictions)),
        'log_loss': float(log_loss(labels, probabilities, labels=list(range(NUM_CLASSES)))),
        'total_errors': int((predictions != labels).sum()),
        'recyclable_to_organic': int(matrix[0, 2]),
        'organic_to_recyclable': int(matrix[2, 0]),
        'ro_errors': int(matrix[0, 2] + matrix[2, 0]),
        'f1_recyclable': float(report['Recyclable']['f1-score']),
        'f1_electronic': float(report['Electronic']['f1-score']),
        'f1_organic': float(report['Organic']['f1-score']),
        'confusion_matrix': matrix.tolist(),
        'classification_report': report,
    }


def prediction_frame(indices, probabilities, candidate_name, stage, fold):
    indices = np.asarray(indices, dtype=np.int64)
    labels = train_df.iloc[indices]['label'].to_numpy(dtype=np.int64)
    predictions = probabilities.argmax(axis=1)
    return pd.DataFrame({
        'row_index': indices,
        'image_id': train_df.iloc[indices]['image_id'].to_numpy(),
        'label': labels,
        'prediction': predictions,
        'prob_0': probabilities[:, 0],
        'prob_1': probabilities[:, 1],
        'prob_2': probabilities[:, 2],
        'correct': predictions == labels,
        'fold': fold,
        'candidate': candidate_name,
        'stage': stage,
    })


def report_frame(metrics):
    rows = []
    for class_name in CLASS_NAMES:
        row = {'label': class_name}
        row.update(metrics['classification_report'][class_name])
        rows.append(row)
    return pd.DataFrame(rows)


def train_cv_head(candidate_name, validation_fold, stage):
    fold_values = cached_folds.numpy()
    train_indices = np.flatnonzero(fold_values != validation_fold)
    valid_indices = np.flatnonzero(fold_values == validation_fold)
    if set(fold_values[train_indices]) != (set(range(N_FOLDS)) - {validation_fold}):
        raise RuntimeError('Training fold membership failed.')
    if set(fold_values[valid_indices]) != {validation_fold}:
        raise RuntimeError('Validation fold membership failed.')
    train_labels = cached_labels[torch.from_numpy(train_indices)].numpy()
    valid_labels = cached_labels[torch.from_numpy(valid_indices)].numpy()
    class_counts, class_weights = class_weights_for(train_labels)

    seed_everything(SEED)
    torch.cuda.reset_peak_memory_stats(HEAD_DEVICE)
    model = build_head(candidate_name).to(HEAD_DEVICE)
    criterion = nn.CrossEntropyLoss(weight=class_weights, label_smoothing=LABEL_SMOOTHING)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=MAX_EPOCHS)
    generator = torch.Generator().manual_seed(SEED)
    loader = DataLoader(
        CachedHeadRows(candidate_name, train_indices),
        batch_size=HEAD_BATCH_SIZE,
        shuffle=True,
        generator=generator,
        num_workers=0,
        pin_memory=True,
    )
    state = {'best_f1': -1.0, 'best_log_loss': float('inf'), 'best_epoch': None, 'state_dict': None, 'stale': 0}
    history = []
    started = time.perf_counter()
    for epoch in range(1, MAX_EPOCHS + 1):
        model.train()
        epoch_loss = 0.0
        seen = 0
        learning_rate = optimizer.param_groups[0]['lr']
        epoch_started = time.perf_counter()
        for features, labels, _indices in loader:
            labels = labels.to(HEAD_DEVICE, non_blocking=True)
            optimizer.zero_grad(set_to_none=True)
            loss = batch_loss(model, candidate_name, features, labels, criterion)
            if not torch.isfinite(loss):
                raise RuntimeError('Non-finite loss for {} fold {} epoch {}.'.format(
                    candidate_name, validation_fold, epoch
                ))
            loss.backward()
            gradient_norm = nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP_MAX_NORM)
            if not torch.isfinite(gradient_norm):
                raise RuntimeError('Non-finite gradient norm.')
            optimizer.step()
            epoch_loss += loss.item() * len(labels)
            seen += len(labels)
        scheduler.step()
        probabilities = predict_probabilities(model, candidate_name, valid_indices)
        metrics = metric_bundle(valid_labels, probabilities)
        history.append({
            'stage': stage,
            'candidate': candidate_name,
            'fold': validation_fold,
            'epoch': epoch,
            'train_loss': epoch_loss / seen,
            'valid_macro_f1': metrics['macro_f1'],
            'valid_log_loss': metrics['log_loss'],
            'learning_rate': learning_rate,
            'epoch_seconds': time.perf_counter() - epoch_started,
        })
        print('{} {} fold={} epoch={:02d} loss={:.6f} f1={:.6f}'.format(
            stage, candidate_name, validation_fold, epoch, epoch_loss / seen, metrics['macro_f1']
        ))
        improved = metrics['macro_f1'] > state['best_f1'] + 1e-8 or (
            abs(metrics['macro_f1'] - state['best_f1']) <= 1e-8 and metrics['log_loss'] < state['best_log_loss']
        )
        if improved:
            state.update({
                'best_f1': metrics['macro_f1'],
                'best_log_loss': metrics['log_loss'],
                'best_epoch': epoch,
                'state_dict': {name: tensor.detach().cpu().clone() for name, tensor in model.state_dict().items()},
                'stale': 0,
            })
        else:
            state['stale'] += 1
        if state['stale'] >= EARLY_STOPPING_PATIENCE:
            break
    if state['state_dict'] is None:
        raise RuntimeError('Training produced no checkpoint.')

    model.load_state_dict(state['state_dict'], strict=True)
    probabilities = predict_probabilities(model, candidate_name, valid_indices)
    metrics = metric_bundle(valid_labels, probabilities)
    metrics.update({
        'stage': stage,
        'candidate': candidate_name,
        'candidate_spec': HEAD_SPECS[candidate_name],
        'validation_fold': validation_fold,
        'best_epoch': state['best_epoch'],
        'seed': SEED,
        'train_class_counts': class_counts.tolist(),
        'training_seconds': time.perf_counter() - started,
        'peak_cuda_0_training_gib': torch.cuda.max_memory_allocated(HEAD_DEVICE) / 2 ** 30,
        'cache_signature': TRAIN_CACHE_SIGNATURE,
        'fold_assignment_signature': FOLD_ASSIGNMENT_SIGNATURE,
        'probabilities_finite': bool(np.isfinite(probabilities).all()),
        'row_order_verified': True,
        'single_model_contract': True,
    })
    frame = prediction_frame(valid_indices, probabilities, candidate_name, stage, validation_fold)
    stem = '{}_{}_fold{}'.format(stage, candidate_name, validation_fold)
    write_csv_atomic(frame, HEADS_DIR / (stem + '_predictions.csv'))
    write_csv_atomic(pd.DataFrame(history), HEADS_DIR / (stem + '_history.csv'))
    write_csv_atomic(report_frame(metrics), HEADS_DIR / (stem + '_classification_report.csv'))
    write_json_atomic(metrics, HEADS_DIR / (stem + '_metrics.json'))
    atomic_torch_save({
        'metadata': {
            'candidate': candidate_name,
            'architecture': HEAD_SPECS[candidate_name]['architecture'],
            'representation': HEAD_SPECS[candidate_name]['representation'],
            'projection_checksum': HEAD_SPECS[candidate_name]['projection_checksum'],
            'cache_signature': TRAIN_CACHE_SIGNATURE,
            'fold_assignment_signature': FOLD_ASSIGNMENT_SIGNATURE,
            'validation_fold': validation_fold,
            'best_epoch': state['best_epoch'],
            'seed': SEED,
        },
        'state_dict': state['state_dict'],
    }, HEADS_DIR / (stem + '.pt'))
    del model, criterion, optimizer, scheduler, loader
    release_cuda_cache()
    return metrics, frame, probabilities


print(json.dumps(HEAD_SPECS, indent=2))


## 9. Fold-0 development, locked assessment, OOF diagnostics, and train-only selection

In [ ]:
CV_METRICS = {CONTROL_NAME: {}, CANDIDATE_NAME: {}}
CV_FRAMES = {CONTROL_NAME: {}, CANDIDATE_NAME: {}}
CV_PROBABILITIES = {CONTROL_NAME: {}, CANDIDATE_NAME: {}}

for candidate_name in (CONTROL_NAME, CANDIDATE_NAME):
    metrics, frame, probabilities = train_cv_head(candidate_name, DEVELOPMENT_FOLD, 'development')
    CV_METRICS[candidate_name][DEVELOPMENT_FOLD] = metrics
    CV_FRAMES[candidate_name][DEVELOPMENT_FOLD] = frame
    CV_PROBABILITIES[candidate_name][DEVELOPMENT_FOLD] = probabilities
    write_csv_atomic(frame, ARTIFACT_DIR / ('development_predictions_' + candidate_name + '.csv'))
    write_npy_atomic(probabilities, ARTIFACT_DIR / ('development_probabilities_' + candidate_name + '.npy'))

development_control = CV_METRICS[CONTROL_NAME][DEVELOPMENT_FOLD]
development_candidate = CV_METRICS[CANDIDATE_NAME][DEVELOPMENT_FOLD]
DEVELOPMENT_GATE_CHECKS = {
    'macro_f1_gain_at_least_0.0010': (
        development_candidate['macro_f1'] - development_control['macro_f1'] >= DEV_MIN_GAIN
    ),
    'errors_reduced_by_at_least_5': (
        development_candidate['total_errors'] <= development_control['total_errors'] - DEV_MIN_ERROR_REDUCTION
    ),
    'ro_errors_reduced_by_at_least_5': (
        development_candidate['ro_errors'] <= development_control['ro_errors'] - DEV_MIN_RO_REDUCTION
    ),
    'electronic_f1_decline_at_most_0.0005': (
        development_control['f1_electronic'] - development_candidate['f1_electronic']
        <= MAX_ELECTRONIC_F1_DROP
    ),
    'probabilities_finite': bool(
        development_control['probabilities_finite'] and development_candidate['probabilities_finite']
    ),
    'row_order_verified': bool(
        development_control['row_order_verified'] and development_candidate['row_order_verified']
    ),
    'single_model_contract': bool(
        development_control['single_model_contract'] and development_candidate['single_model_contract']
    ),
}
DEVELOPMENT_GATE_PASSED = all(DEVELOPMENT_GATE_CHECKS.values())
DEVELOPMENT_SUMMARY = {
    'control': development_control,
    'candidate': development_candidate,
    'candidate_minus_control_macro_f1': development_candidate['macro_f1'] - development_control['macro_f1'],
    'candidate_minus_control_errors': development_candidate['total_errors'] - development_control['total_errors'],
    'candidate_minus_control_ro_errors': development_candidate['ro_errors'] - development_control['ro_errors'],
    'development_gate_checks': DEVELOPMENT_GATE_CHECKS,
    'development_gate_passed': DEVELOPMENT_GATE_PASSED,
}
write_json_atomic(DEVELOPMENT_SUMMARY, ARTIFACT_DIR / 'development_metrics.json')
write_csv_atomic(pd.DataFrame([
    {
        'candidate': name,
        'fold': DEVELOPMENT_FOLD,
        'macro_f1': CV_METRICS[name][0]['macro_f1'],
        'total_errors': CV_METRICS[name][0]['total_errors'],
        'ro_errors': CV_METRICS[name][0]['ro_errors'],
        'f1_electronic': CV_METRICS[name][0]['f1_electronic'],
        'best_epoch': CV_METRICS[name][0]['best_epoch'],
    }
    for name in (CONTROL_NAME, CANDIDATE_NAME)
]), ARTIFACT_DIR / 'development_metrics.csv')

LOCKED_RECIPE = {
    'locked_before_assessment': True,
    'revision_marker': REVISION_MARKER,
    'config_signature': CONFIG_SIGNATURE,
    'data_contract_signature': DATA_CONTRACT_SIGNATURE,
    'fold_assignment_signature': FOLD_ASSIGNMENT_SIGNATURE,
    'train_cache_signature': TRAIN_CACHE_SIGNATURE,
    'projection_checksum': PROJECTION_CHECKSUM,
    'control_spec': HEAD_SPECS[CONTROL_NAME],
    'candidate_spec': HEAD_SPECS[CANDIDATE_NAME],
    'views': VIEW_CONFIG,
    'view_ce_weights': VIEW_CE_WEIGHTS,
    'consistency_weight': CONSISTENCY_WEIGHT,
    'optimizer': 'AdamW',
    'scheduler': 'CosineAnnealingLR(T_max=15)',
    'seed': SEED,
    'max_epochs': MAX_EPOCHS,
    'patience': EARLY_STOPPING_PATIENCE,
    'learning_rate': LEARNING_RATE,
    'weight_decay': WEIGHT_DECAY,
    'label_smoothing': LABEL_SMOOTHING,
    'gradient_clip': GRAD_CLIP_MAX_NORM,
    'development_thresholds': CONFIG['development_gate'],
    'adoption_thresholds': CONFIG['adoption_gate'],
    'development_gate_checks': DEVELOPMENT_GATE_CHECKS,
    'candidate_assessment_unlocked': DEVELOPMENT_GATE_PASSED,
    'assessment_folds': list(ASSESSMENT_FOLDS),
    'recipe_changes_after_lock_allowed': False,
    'test_images_accessed': False,
}
LOCKED_RECIPE_SIGNATURE = sha256_json(LOCKED_RECIPE)
LOCKED_RECIPE['locked_recipe_signature'] = LOCKED_RECIPE_SIGNATURE
write_json_atomic(LOCKED_RECIPE, ARTIFACT_DIR / 'locked_recipe_before_assessment.json')

# Control folds are always measured as the reproducibility/fallback reference.
# Candidate folds 1–4 are physically unreachable when the locked fold-0 gate fails.
for fold in ASSESSMENT_FOLDS:
    metrics, frame, probabilities = train_cv_head(CONTROL_NAME, fold, 'assessment')
    CV_METRICS[CONTROL_NAME][fold] = metrics
    CV_FRAMES[CONTROL_NAME][fold] = frame
    CV_PROBABILITIES[CONTROL_NAME][fold] = probabilities
if DEVELOPMENT_GATE_PASSED:
    for fold in ASSESSMENT_FOLDS:
        metrics, frame, probabilities = train_cv_head(CANDIDATE_NAME, fold, 'assessment')
        CV_METRICS[CANDIDATE_NAME][fold] = metrics
        CV_FRAMES[CANDIDATE_NAME][fold] = frame
        CV_PROBABILITIES[CANDIDATE_NAME][fold] = probabilities


def build_oof_artifacts(candidate_name):
    if set(CV_PROBABILITIES[candidate_name]) != set(range(N_FOLDS)):
        raise RuntimeError('{} does not have five complete OOF folds.'.format(candidate_name))
    probabilities = np.full((len(train_df), NUM_CLASSES), np.nan, dtype=np.float64)
    frames = []
    for fold in range(N_FOLDS):
        frame = CV_FRAMES[candidate_name][fold]
        indices = frame['row_index'].to_numpy(dtype=np.int64)
        probabilities[indices] = CV_PROBABILITIES[candidate_name][fold]
        frames.append(frame)
    if not np.isfinite(probabilities).all():
        raise RuntimeError('{} OOF matrix has missing/non-finite rows.'.format(candidate_name))
    oof_frame = pd.concat(frames, ignore_index=True).sort_values('row_index').reset_index(drop=True)
    if oof_frame['row_index'].tolist() != list(range(len(train_df))):
        raise RuntimeError('{} OOF row order is incomplete.'.format(candidate_name))
    labels = train_df['label'].to_numpy(dtype=np.int64)
    metrics = metric_bundle(labels, probabilities)
    metrics.update({
        'candidate': candidate_name,
        'fold_assignment_signature': FOLD_ASSIGNMENT_SIGNATURE,
        'cache_signature': TRAIN_CACHE_SIGNATURE,
        'fold_best_epochs': [CV_METRICS[candidate_name][fold]['best_epoch'] for fold in range(N_FOLDS)],
        'probabilities_finite': True,
        'row_order_verified': True,
    })
    write_csv_atomic(oof_frame, ARTIFACT_DIR / ('oof_predictions_' + candidate_name + '.csv'))
    write_npy_atomic(probabilities, ARTIFACT_DIR / ('oof_probabilities_' + candidate_name + '.npy'))
    write_json_atomic(metrics, ARTIFACT_DIR / ('oof_metrics_' + candidate_name + '.json'))
    raw = np.asarray(metrics['confusion_matrix'], dtype=np.int64)
    normalized = raw / np.maximum(raw.sum(axis=1, keepdims=True), 1)
    write_csv_atomic(pd.DataFrame(raw, index=CLASS_NAMES, columns=CLASS_NAMES),
                     ARTIFACT_DIR / ('confusion_matrix_raw_' + candidate_name + '.csv'), index=True)
    write_csv_atomic(pd.DataFrame(normalized, index=CLASS_NAMES, columns=CLASS_NAMES),
                     ARTIFACT_DIR / ('confusion_matrix_normalized_' + candidate_name + '.csv'), index=True)
    write_csv_atomic(report_frame(metrics), ARTIFACT_DIR / ('classification_report_' + candidate_name + '.csv'))
    ro_errors = oof_frame[
        ((oof_frame['label'] == 0) & (oof_frame['prediction'] == 2))
        | ((oof_frame['label'] == 2) & (oof_frame['prediction'] == 0))
    ].copy()
    write_csv_atomic(ro_errors, ARTIFACT_DIR / ('oof_recyclable_organic_errors_' + candidate_name + '.csv'))
    return metrics, oof_frame, probabilities


OOF_METRICS = {}
OOF_FRAMES = {}
OOF_PROBABILITIES = {}
OOF_METRICS[CONTROL_NAME], OOF_FRAMES[CONTROL_NAME], OOF_PROBABILITIES[CONTROL_NAME] = build_oof_artifacts(
    CONTROL_NAME
)
if DEVELOPMENT_GATE_PASSED:
    OOF_METRICS[CANDIDATE_NAME], OOF_FRAMES[CANDIDATE_NAME], OOF_PROBABILITIES[CANDIDATE_NAME] = (
        build_oof_artifacts(CANDIDATE_NAME)
    )

assessment_rows = []
for candidate_name in (CONTROL_NAME, CANDIDATE_NAME):
    for fold in ASSESSMENT_FOLDS:
        if fold not in CV_METRICS[candidate_name]:
            continue
        metrics = CV_METRICS[candidate_name][fold]
        assessment_rows.append({
            'candidate': candidate_name,
            'fold': fold,
            'macro_f1': metrics['macro_f1'],
            'total_errors': metrics['total_errors'],
            'ro_errors': metrics['ro_errors'],
            'f1_electronic': metrics['f1_electronic'],
            'best_epoch': metrics['best_epoch'],
        })
assessment_frame = pd.DataFrame(assessment_rows)
write_csv_atomic(assessment_frame, ARTIFACT_DIR / 'assessment_per_fold.csv')
aggregate_rows = []
for candidate_name in (CONTROL_NAME, CANDIDATE_NAME):
    candidate_rows = assessment_frame[assessment_frame['candidate'] == candidate_name]
    if candidate_rows.empty:
        aggregate_rows.append({'candidate': candidate_name, 'assessed': False, 'reason': 'fold-0 gate failed'})
        continue
    aggregate_rows.append({
        'candidate': candidate_name,
        'assessed': True,
        'assessment_folds': len(candidate_rows),
        'assessment_macro_f1_mean': candidate_rows['macro_f1'].mean(),
        'assessment_macro_f1_std': candidate_rows['macro_f1'].std(ddof=0),
        'five_fold_oof_macro_f1': OOF_METRICS.get(candidate_name, {}).get('macro_f1'),
        'five_fold_errors': OOF_METRICS.get(candidate_name, {}).get('total_errors'),
        'five_fold_ro_errors': OOF_METRICS.get(candidate_name, {}).get('ro_errors'),
    })
write_csv_atomic(pd.DataFrame(aggregate_rows), ARTIFACT_DIR / 'assessment_aggregate.csv')

if DEVELOPMENT_GATE_PASSED:
    control_oof = OOF_METRICS[CONTROL_NAME]
    candidate_oof = OOF_METRICS[CANDIDATE_NAME]
    fold_gains = [
        CV_METRICS[CANDIDATE_NAME][fold]['macro_f1'] - CV_METRICS[CONTROL_NAME][fold]['macro_f1']
        for fold in range(N_FOLDS)
    ]
    positive_gains = [max(gain, 0.0) for gain in fold_gains]
    positive_gain_sum = sum(positive_gains)
    largest_positive_share = max(positive_gains) / positive_gain_sum if positive_gain_sum > 0 else 1.0
    ADOPTION_GATE_CHECKS = {
        'development_gate_passed': True,
        'oof_macro_f1_at_least_0.9911': candidate_oof['macro_f1'] >= ADOPT_MIN_OOF_F1,
        'gain_over_matched_control_at_least_0.0010': (
            candidate_oof['macro_f1'] - control_oof['macro_f1'] >= ADOPT_MIN_GAIN
        ),
        'total_errors_at_most_282': candidate_oof['total_errors'] <= ADOPT_MAX_ERRORS,
        'ro_errors_at_most_237': candidate_oof['ro_errors'] <= ADOPT_MAX_RO_ERRORS,
        'wins_at_least_3_folds': sum(gain > 0 for gain in fold_gains) >= ADOPT_MIN_FOLD_WINS,
        'not_one_fold_dominated': largest_positive_share <= ADOPT_MAX_SINGLE_FOLD_GAIN_SHARE,
        'electronic_f1_decline_at_most_0.0005': (
            control_oof['f1_electronic'] - candidate_oof['f1_electronic'] <= MAX_ELECTRONIC_F1_DROP
        ),
        'finite_and_ordered': bool(
            candidate_oof['probabilities_finite'] and candidate_oof['row_order_verified']
        ),
        'no_test_leakage_or_prediction_overrides': bool(
            not TEST_ACCESS_STATE['test_images_accessed'] and not TEST_ACCESS_STATE['manual_test_labels_accessed']
        ),
        'one_model_contract': True,
    }
else:
    fold_gains = []
    largest_positive_share = None
    ADOPTION_GATE_CHECKS = {
        'development_gate_passed': False,
        'candidate_assessment_not_run': set(CV_METRICS[CANDIDATE_NAME]) == {DEVELOPMENT_FOLD},
    }

CANDIDATE_ADOPTED = bool(DEVELOPMENT_GATE_PASSED and all(ADOPTION_GATE_CHECKS.values()))
SELECTED_CANDIDATE = CANDIDATE_NAME if CANDIDATE_ADOPTED else CONTROL_NAME
SELECTION_DECISION = {
    'decision_locked_from_train_only_evidence': True,
    'selected_candidate': SELECTED_CANDIDATE,
    'candidate_adopted': CANDIDATE_ADOPTED,
    'candidate_status': 'adopted' if CANDIDATE_ADOPTED else 'rejected',
    'development_gate_passed': DEVELOPMENT_GATE_PASSED,
    'development_thresholds': CONFIG['development_gate'],
    'development_gate_checks': DEVELOPMENT_GATE_CHECKS,
    'adoption_thresholds': CONFIG['adoption_gate'],
    'adoption_gate_checks': ADOPTION_GATE_CHECKS,
    'fold_macro_f1_gains': fold_gains,
    'largest_positive_fold_gain_share': largest_positive_share,
    'control_oof_metrics': OOF_METRICS[CONTROL_NAME],
    'candidate_oof_metrics': OOF_METRICS.get(CANDIDATE_NAME),
    'locked_recipe_signature': LOCKED_RECIPE_SIGNATURE,
    'config_signature': CONFIG_SIGNATURE,
    'data_contract_signature': DATA_CONTRACT_SIGNATURE,
    'fold_assignment_signature': FOLD_ASSIGNMENT_SIGNATURE,
    'train_cache_signature': TRAIN_CACHE_SIGNATURE,
    'projection_checksum': PROJECTION_CHECKSUM,
    'test_images_accessed_at_decision': False,
    'external_data_used': False,
    'row_overrides': False,
    'threshold_adjustment': False,
    'blend_or_ensemble': False,
    'decision_overrides_allowed': False,
}
SELECTION_SIGNATURE = sha256_json(SELECTION_DECISION)
SELECTION_DECISION['selection_signature'] = SELECTION_SIGNATURE
write_json_atomic(SELECTION_DECISION, SELECTION_DECISION_PATH)


def query_attention_diagnostics(folds):
    map_sums = np.zeros((NUM_CLASSES, MATERIAL_QUERIES, SPATIAL_TOKENS), dtype=np.float64)
    entropy_sums = np.zeros((NUM_CLASSES, MATERIAL_QUERIES), dtype=np.float64)
    max_sums = np.zeros((NUM_CLASSES, MATERIAL_QUERIES), dtype=np.float64)
    counts = np.zeros(NUM_CLASSES, dtype=np.int64)
    for fold in folds:
        stage = 'development' if fold == DEVELOPMENT_FOLD else 'assessment'
        checkpoint_path = HEADS_DIR / ('{}_{}_fold{}.pt'.format(stage, CANDIDATE_NAME, fold))
        checkpoint = torch.load(checkpoint_path, map_location='cpu', weights_only=False)
        if checkpoint['metadata']['projection_checksum'] != PROJECTION_CHECKSUM:
            raise RuntimeError('Attention checkpoint projection signature mismatch.')
        model = build_head(CANDIDATE_NAME).to(HEAD_DEVICE)
        model.load_state_dict(checkpoint['state_dict'], strict=True)
        valid_indices = np.flatnonzero(cached_folds.numpy() == fold)
        loader = DataLoader(
            CachedHeadRows(CANDIDATE_NAME, valid_indices),
            batch_size=PREDICT_BATCH_SIZE,
            shuffle=False,
            num_workers=0,
            pin_memory=True,
        )
        model.eval()
        with torch.inference_mode():
            for features, labels, _indices in loader:
                _logits, weights = model(
                    features['original'].to(HEAD_DEVICE, dtype=torch.float32, non_blocking=True),
                    return_attention=True,
                )
                weights = weights.mean(dim=1).float().cpu().numpy()
                labels = labels.numpy()
                entropy = -(weights * np.log(np.clip(weights, 1e-12, None))).sum(axis=2) / math.log(SPATIAL_TOKENS)
                maxima = weights.max(axis=2)
                for class_index in range(NUM_CLASSES):
                    mask = labels == class_index
                    if mask.any():
                        map_sums[class_index] += weights[mask].sum(axis=0)
                        entropy_sums[class_index] += entropy[mask].sum(axis=0)
                        max_sums[class_index] += maxima[mask].sum(axis=0)
                        counts[class_index] += int(mask.sum())
        del model, loader, checkpoint
        release_cuda_cache()
    if (counts == 0).any():
        raise RuntimeError('Attention diagnostics are missing a class.')
    average_maps = map_sums / counts[:, None, None]
    map_rows = []
    summary_rows = []
    coordinates = np.arange(SPATIAL_TOKENS)
    patch_rows = coordinates // POOLED_GRID
    patch_columns = coordinates % POOLED_GRID
    for class_index, class_name in enumerate(CLASS_NAMES):
        for query_index in range(MATERIAL_QUERIES):
            attention_map = average_maps[class_index, query_index]
            top_index = int(attention_map.argmax())
            summary_rows.append({
                'true_class': class_name,
                'query': query_index,
                'rows': int(counts[class_index]),
                'mean_normalized_entropy': entropy_sums[class_index, query_index] / counts[class_index],
                'mean_max_attention': max_sums[class_index, query_index] / counts[class_index],
                'top_patch_row': top_index // POOLED_GRID,
                'top_patch_column': top_index % POOLED_GRID,
                'center_of_mass_row': float((attention_map * patch_rows).sum()),
                'center_of_mass_column': float((attention_map * patch_columns).sum()),
            })
            for patch_index, weight in enumerate(attention_map):
                map_rows.append({
                    'true_class': class_name,
                    'query': query_index,
                    'patch_row': patch_index // POOLED_GRID,
                    'patch_column': patch_index % POOLED_GRID,
                    'mean_attention': float(weight),
                })
    global_maps = map_sums.sum(axis=0) / counts.sum()
    normalized_maps = global_maps / np.maximum(np.linalg.norm(global_maps, axis=1, keepdims=True), 1e-12)
    query_cosine = normalized_maps @ normalized_maps.T
    off_diagonal = query_cosine[~np.eye(MATERIAL_QUERIES, dtype=bool)]
    payload = {
        'folds': list(folds),
        'official_train_validation_only': True,
        'test_labels_used': False,
        'selection_gate_uses_diagnostics': False,
        'parameter_count': HEAD_SPECS[CANDIDATE_NAME]['parameter_count'],
        'peak_cuda_0_training_gib_by_fold': {
            str(fold): CV_METRICS[CANDIDATE_NAME][fold]['peak_cuda_0_training_gib'] for fold in folds
        },
        'attention_shape': ['batch', ATTENTION_HEADS, MATERIAL_QUERIES, SPATIAL_TOKENS],
        'query_pair_cosine_matrix': query_cosine.tolist(),
        'maximum_off_diagonal_query_cosine': float(off_diagonal.max()),
        'possible_query_collapse_diagnostic': bool(off_diagonal.max() > 0.98),
        'summary': summary_rows,
    }
    write_csv_atomic(pd.DataFrame(map_rows), ARTIFACT_DIR / 'query_attention_maps_by_true_class.csv')
    write_csv_atomic(pd.DataFrame(summary_rows), ARTIFACT_DIR / 'query_attention_diagnostics.csv')
    write_json_atomic(payload, ARTIFACT_DIR / 'query_attention_diagnostics.json')
    return payload


ATTENTION_DIAGNOSTICS = query_attention_diagnostics(sorted(CV_METRICS[CANDIDATE_NAME]))
ERROR_ANALYSIS = {
    'selected_candidate': SELECTED_CANDIDATE,
    'selected_oof_metrics': OOF_METRICS[SELECTED_CANDIDATE],
    'control_oof_metrics': OOF_METRICS[CONTROL_NAME],
    'candidate_oof_metrics': OOF_METRICS.get(CANDIDATE_NAME),
    'qualitative_failure_modes_are_hypotheses_not_labels': True,
    'row_level_overrides': False,
}
write_json_atomic(ERROR_ANALYSIS, ARTIFACT_DIR / 'error_analysis.json')
print('Train-only selection:', SELECTED_CANDIDATE, '| candidate adopted:', CANDIDATE_ADOPTED)


## 10. Full-train selected head

The selected recipe is retrained once from scratch on all official training rows. Epoch count is the rounded median of its five measured fold-best epochs. If V11 fails fold 0, the exact control is the only final option and candidate assessment is never run.

In [ ]:
persisted_selection = read_json(SELECTION_DECISION_PATH)
persisted_signature = persisted_selection.pop('selection_signature')
if sha256_json(persisted_selection) != persisted_signature or persisted_signature != SELECTION_SIGNATURE:
    raise RuntimeError('Persisted train-only selection signature is invalid.')
persisted_selection['selection_signature'] = persisted_signature
if persisted_selection['selected_candidate'] != SELECTED_CANDIDATE:
    raise RuntimeError('In-memory and persisted selected recipes differ.')
if TEST_ACCESS_STATE['test_images_accessed']:
    raise RuntimeError('Test access occurred before full-data training.')

SELECTED_BEST_EPOCHS = [
    int(CV_METRICS[SELECTED_CANDIDATE][fold]['best_epoch']) for fold in range(N_FOLDS)
]
FINAL_EPOCHS = int(round(statistics.median(SELECTED_BEST_EPOCHS)))
FINAL_EPOCHS = max(1, min(MAX_EPOCHS, FINAL_EPOCHS))
all_indices = np.arange(len(train_df), dtype=np.int64)
class_counts, class_weights = class_weights_for(cached_labels.numpy())

seed_everything(SEED)
torch.cuda.reset_peak_memory_stats(HEAD_DEVICE)
FINAL_MODEL = build_head(SELECTED_CANDIDATE).to(HEAD_DEVICE)
criterion = nn.CrossEntropyLoss(weight=class_weights, label_smoothing=LABEL_SMOOTHING)
optimizer = torch.optim.AdamW(FINAL_MODEL.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=MAX_EPOCHS)
generator = torch.Generator().manual_seed(SEED)
full_loader = DataLoader(
    CachedHeadRows(SELECTED_CANDIDATE, all_indices),
    batch_size=HEAD_BATCH_SIZE,
    shuffle=True,
    generator=generator,
    num_workers=0,
    pin_memory=True,
)
FINAL_HISTORY = []
full_training_started = time.perf_counter()
for epoch in range(1, FINAL_EPOCHS + 1):
    FINAL_MODEL.train()
    epoch_loss = 0.0
    seen = 0
    learning_rate = optimizer.param_groups[0]['lr']
    epoch_started = time.perf_counter()
    for features, labels, _indices in full_loader:
        labels = labels.to(HEAD_DEVICE, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        loss = batch_loss(FINAL_MODEL, SELECTED_CANDIDATE, features, labels, criterion)
        if not torch.isfinite(loss):
            raise RuntimeError('Non-finite full-training loss at epoch {}.'.format(epoch))
        loss.backward()
        gradient_norm = nn.utils.clip_grad_norm_(FINAL_MODEL.parameters(), GRAD_CLIP_MAX_NORM)
        if not torch.isfinite(gradient_norm):
            raise RuntimeError('Non-finite full-training gradient norm.')
        optimizer.step()
        epoch_loss += loss.item() * len(labels)
        seen += len(labels)
    scheduler.step()
    FINAL_HISTORY.append({
        'candidate': SELECTED_CANDIDATE,
        'epoch': epoch,
        'epochs_planned': FINAL_EPOCHS,
        'train_loss': epoch_loss / seen,
        'learning_rate': learning_rate,
        'epoch_seconds': time.perf_counter() - epoch_started,
    })
    print('full {} epoch={:02d}/{:02d} loss={:.6f}'.format(
        SELECTED_CANDIDATE, epoch, FINAL_EPOCHS, epoch_loss / seen
    ))

FINAL_MODEL.eval()
FINAL_HEAD_METADATA = {
    'candidate': SELECTED_CANDIDATE,
    'candidate_spec': HEAD_SPECS[SELECTED_CANDIDATE],
    'architecture_name': SELECTED_CANDIDATE,
    'representation_name': HEAD_SPECS[SELECTED_CANDIDATE]['representation'],
    'branch_or_token_names': (
        ['final_cls', 'final_true_patch_mean']
        if SELECTED_CANDIDATE == CONTROL_NAME
        else ['pooled_patch_8x8_row_major', 'projected_cls', 'projected_true_patch_mean']
    ),
    'selection_signature': SELECTION_SIGNATURE,
    'locked_recipe_signature': LOCKED_RECIPE_SIGNATURE,
    'config_signature': CONFIG_SIGNATURE,
    'data_contract_signature': DATA_CONTRACT_SIGNATURE,
    'fold_assignment_signature': FOLD_ASSIGNMENT_SIGNATURE,
    'train_cache_signature': TRAIN_CACHE_SIGNATURE,
    'projection_checksum': PROJECTION_CHECKSUM if SELECTED_CANDIDATE == CANDIDATE_NAME else None,
    'projection_state_file': PROJECTION_STATE_PATH.name,
    'full_train_rows': len(train_df),
    'fold_best_epochs': SELECTED_BEST_EPOCHS,
    'epoch_rule': 'rounded median of five selected-recipe CV best epochs',
    'epochs': FINAL_EPOCHS,
    'seed': SEED,
    'class_counts': class_counts.tolist(),
    'training_seconds': time.perf_counter() - full_training_started,
    'peak_cuda_0_training_gib': torch.cuda.max_memory_allocated(HEAD_DEVICE) / 2 ** 30,
    'one_head': True,
    'fold_checkpoint_ensemble': False,
    'test_images_accessed_before_training': False,
}
FINAL_HEAD_PATH = ARTIFACT_DIR / 'final_head.pt'
atomic_torch_save({
    'metadata': FINAL_HEAD_METADATA,
    'state_dict': {name: tensor.detach().cpu().clone() for name, tensor in FINAL_MODEL.state_dict().items()},
}, FINAL_HEAD_PATH)
FINAL_HEAD_SHA256 = sha256_file(FINAL_HEAD_PATH)
write_csv_atomic(pd.DataFrame(FINAL_HISTORY), ARTIFACT_DIR / 'final_training_history.csv')
write_json_atomic({**FINAL_HEAD_METADATA, 'checkpoint_sha256': FINAL_HEAD_SHA256},
                  ARTIFACT_DIR / 'final_head_metadata.json')
del criterion, optimizer, scheduler, full_loader
release_cuda_cache()
print('Final head saved:', FINAL_HEAD_PATH, '| epochs:', FINAL_EPOCHS)


## 11. Post-lock official test inference

This section is the first code allowed to discover the test directory or submission template. It reloads the persisted train-only decision, verifies recipe signatures, extracts only the original deterministic view, runs the one selected head once, writes direct probabilities and argmax labels in exact template order, then reconstructs the head/projection for checkpoint reproduction. No TTA, folds/seeds ensemble, blend, routing, threshold shift, or row correction exists.

In [ ]:
persisted_selection = read_json(SELECTION_DECISION_PATH)
persisted_selection_signature = persisted_selection.pop('selection_signature', None)
persisted_lock = read_json(ARTIFACT_DIR / 'locked_recipe_before_assessment.json')
persisted_lock_signature = persisted_lock.pop('locked_recipe_signature', None)
persisted_config = read_json(ARTIFACT_DIR / 'config.json')
persisted_data_contract = read_json(ARTIFACT_DIR / 'data_contract.json')
persisted_fold_signature = read_json(ARTIFACT_DIR / 'fold_signature.json')
persisted_train_cache_manifest = read_json(ARTIFACT_DIR / 'cache_manifest_train.json')
TEST_UNLOCK_CHECKS = {
    'selection_signature_valid': bool(
        persisted_selection_signature == SELECTION_SIGNATURE
        and sha256_json(persisted_selection) == persisted_selection_signature
    ),
    'selected_candidate_matches': persisted_selection.get('selected_candidate') == SELECTED_CANDIDATE,
    'selection_train_only': persisted_selection.get('test_images_accessed_at_decision') is False,
    'locked_recipe_matches': bool(
        persisted_lock_signature == LOCKED_RECIPE_SIGNATURE
        and sha256_json(persisted_lock) == persisted_lock_signature
    ),
    'config_signature_matches': persisted_selection.get('config_signature') == CONFIG_SIGNATURE,
    'persisted_config_matches': bool(
        persisted_config.get('modeling_config_signature') == CONFIG_SIGNATURE
        and sha256_json({
            key: value for key, value in persisted_config.items()
            if key not in {'team_name', 'modeling_config_signature'}
        }) == CONFIG_SIGNATURE
    ),
    'persisted_data_contract_matches': sha256_json(persisted_data_contract) == DATA_CONTRACT_SIGNATURE,
    'fold_signature_matches': persisted_selection.get('fold_assignment_signature') == FOLD_ASSIGNMENT_SIGNATURE,
    'persisted_fold_signature_matches': (
        persisted_fold_signature.get('fold_assignment_sha256') == FOLD_ASSIGNMENT_SIGNATURE
    ),
    'train_cache_signature_matches': persisted_selection.get('train_cache_signature') == TRAIN_CACHE_SIGNATURE,
    'train_cache_manifest_signature_matches': (
        sha256_json(persisted_train_cache_manifest.get('cache_identity')) == TRAIN_CACHE_SIGNATURE
    ),
    'train_cache_complete': (
        persisted_train_cache_manifest.get('complete') is True
        and persisted_train_cache_manifest.get('rows') == EXPECTED_TRAIN_ROWS
    ),
    'five_cv_epochs_available': len(SELECTED_BEST_EPOCHS) == N_FOLDS,
    'final_head_exists_and_matches': FINAL_HEAD_PATH.is_file() and sha256_file(FINAL_HEAD_PATH) == FINAL_HEAD_SHA256,
    'manual_test_labels_untouched': not TEST_ACCESS_STATE['manual_test_labels_accessed'],
}
failed_unlock_checks = [name for name, passed in TEST_UNLOCK_CHECKS.items() if not passed]
if failed_unlock_checks:
    raise RuntimeError('Test isolation remains locked; failed: {}.'.format(failed_unlock_checks))
TEST_ACCESS_STATE['unlocked'] = True


def discover_named_test_dirs(base, max_depth=3):
    found = []
    stack = [(base, 0)]
    while stack:
        current, depth = stack.pop()
        try:
            children = list(current.iterdir())
        except OSError:
            continue
        for child in children:
            if not child.is_dir():
                continue
            if child.name.lower() == 'test':
                found.append(child)
            elif depth < max_depth:
                stack.append((child, depth + 1))
    return found


def valid_test_dir(path):
    try:
        entries = list(path.iterdir())
    except OSError:
        return False
    return len(entries) == EXPECTED_TEST_ROWS and all(
        entry.is_file() and not entry.is_symlink() and entry.suffix.lower() in ALLOWED_IMAGE_SUFFIXES
        for entry in entries
    )


def resolve_test_dir():
    matches = {
        path.resolve() for path in discover_named_test_dirs(KAGGLE_INPUT) if valid_test_dir(path)
    }
    if len(matches) != 1:
        raise RuntimeError('Expected one official {}-image test directory; found {}: {}.'.format(
            EXPECTED_TEST_ROWS, len(matches), sorted(map(str, matches))
        ))
    return matches.pop()


def resolve_submission_template():
    matches = {}
    relative_test = TEST_DIR.relative_to(KAGGLE_INPUT.resolve())
    official_input_root = KAGGLE_INPUT / relative_test.parts[0]
    for path in official_input_root.rglob('*.csv'):
        lowered = path.name.lower().replace('-', '_')
        if not any(token in lowered for token in ('submission', 'sample', 'format')):
            continue
        if any(token in lowered for token in ('label', 'truth', 'annotation', 'manual')):
            continue
        try:
            frame = pd.read_csv(path, dtype=str)
        except Exception:
            continue
        if tuple(frame.columns) == SUBMISSION_COLUMNS and len(frame) == EXPECTED_TEST_ROWS:
            matches[str(path.resolve())] = frame
    if len(matches) != 1:
        raise RuntimeError('Expected one official submission template; found {}: {}.'.format(
            len(matches), sorted(matches)
        ))
    path = next(iter(matches))
    return Path(path), matches[path]


TEST_DIR = resolve_test_dir()
TEMPLATE_PATH, submission_template = resolve_submission_template()
TEST_ACCESS_STATE['test_images_accessed'] = True
stem_to_path = {}
for path in sorted(TEST_DIR.iterdir(), key=lambda value: value.name):
    if path.stem in stem_to_path:
        raise RuntimeError('Duplicate official test image stem: {}'.format(path.stem))
    stem_to_path[path.stem] = path
template_ids = submission_template['id'].tolist()
if len(set(map(str, template_ids))) != EXPECTED_TEST_ROWS:
    raise RuntimeError('Submission template ids are not unique.')
ordered_paths = []
for raw_id in template_ids:
    key = Path(str(raw_id)).stem
    if key not in stem_to_path:
        raise RuntimeError('Template id {!r} has no matching test image.'.format(raw_id))
    ordered_paths.append(str(stem_to_path[key].resolve()))
if len(set(ordered_paths)) != EXPECTED_TEST_ROWS:
    raise RuntimeError('Template order did not map to distinct test images.')
test_df = pd.DataFrame({'image_id': [str(value) for value in template_ids], 'path': ordered_paths})
test_df['file_size_bytes'] = [Path(path).stat().st_size for path in test_df['path']]
TEST_ORDER_SIGNATURE = sha256_json(test_df['image_id'].tolist())
TEST_SOURCE_SIGNATURE = sha256_json(test_df[['image_id', 'file_size_bytes']].to_dict('records'))


class OriginalTestRows(Dataset):
    def __init__(self, frame):
        self.paths = frame['path'].tolist()
        self.image_ids = frame['image_id'].tolist()

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, index):
        image = load_rgb_image(self.paths[index])
        return make_view(image, self.image_ids[index], 'original')


def extract_selected_representation(model_input):
    with torch.inference_mode():
        prefix, patches = normalized_final_tokens(
            model_input.to(PATCH_INPUT_DEVICE, dtype=BACKBONE_DTYPE, non_blocking=True)
        )
        cls = prefix[:, 0].float()
        patch_mean = patches.float().mean(dim=1)
        if SELECTED_CANDIDATE == CONTROL_NAME:
            features = torch.cat((cls, patch_mean), dim=1)
        else:
            pooled = patches.float().reshape(
                -1, PATCH_GRID, PATCH_GRID, EXPECTED_HIDDEN_SIZE
            ).reshape(
                -1, POOLED_GRID, 2, POOLED_GRID, 2, EXPECTED_HIDDEN_SIZE
            ).mean(dim=(2, 4)).reshape(-1, SPATIAL_TOKENS, EXPECTED_HIDDEN_SIZE)
            features = torch.cat((
                pooled @ PROJECTION_GPU,
                (cls @ PROJECTION_GPU).unsqueeze(1),
                (patch_mean @ PROJECTION_GPU).unsqueeze(1),
            ), dim=1)
        if not torch.isfinite(features).all():
            raise RuntimeError('Selected test representation contains NaN or Inf.')
        return features.half().cpu()


TEST_FEATURE_SHAPE = [CONTROL_DIM] if SELECTED_CANDIDATE == CONTROL_NAME else [CANDIDATE_TOKENS, PROJECTION_DIM]
TEST_CACHE_IDENTITY = {
    'schema_version': 1,
    'selected_candidate': SELECTED_CANDIDATE,
    'representation': HEAD_SPECS[SELECTED_CANDIDATE]['representation'],
    'model_revision': RESOLVED_REVISION,
    'selection_signature': SELECTION_SIGNATURE,
    'locked_recipe_signature': LOCKED_RECIPE_SIGNATURE,
    'test_order_signature': TEST_ORDER_SIGNATURE,
    'test_source_signature': TEST_SOURCE_SIGNATURE,
    'view': 'original',
    'preprocess_signature': PREPROCESS_SIGNATURE,
    'view_signature': VIEW_SIGNATURE,
    'projection_checksum': PROJECTION_CHECKSUM if SELECTED_CANDIDATE == CANDIDATE_NAME else None,
    'feature_shape_per_row': TEST_FEATURE_SHAPE,
    'storage_dtype': 'float16',
    'inference_backbone_passes_per_image': 1,
}
TEST_CACHE_SIGNATURE = sha256_json(TEST_CACHE_IDENTITY)


def test_shard_path(start, stop):
    return TEST_CACHE_DIR / 'test_{:06d}_{:06d}.pt'.format(start, stop)


def validate_test_shard(path, start, stop):
    payload = torch.load(path, map_location='cpu', weights_only=False)
    expected_shape = (stop - start, *TEST_FEATURE_SHAPE)
    checks = [
        payload.get('metadata', {}).get('cache_signature') == TEST_CACHE_SIGNATURE,
        payload.get('metadata', {}).get('start') == start,
        payload.get('metadata', {}).get('stop') == stop,
        payload.get('metadata', {}).get('ids_signature') == sha256_json(test_df.iloc[start:stop]['image_id'].tolist()),
        tuple(payload['features'].shape) == expected_shape,
        payload['features'].dtype == torch.float16,
        bool(torch.isfinite(payload['features']).all()),
        bool(torch.equal(payload['row_indices'], torch.arange(start, stop))),
    ]
    if not all(checks):
        raise RuntimeError('Existing test cache shard failed its locked signature: {}.'.format(path))
    return payload


test_ranges = expected_shard_ranges(len(test_df), FEATURE_SHARD_SIZE)
test_dataset = OriginalTestRows(test_df)
for start, stop in tqdm(test_ranges, desc='original-view test cache'):
    path = test_shard_path(start, stop)
    if path.is_file():
        validate_test_shard(path, start, stop)
        continue
    subset = torch.utils.data.Subset(test_dataset, range(start, stop))
    loader = DataLoader(
        subset,
        batch_size=FEATURE_BATCH_SIZE,
        shuffle=False,
        num_workers=FEATURE_LOADER_WORKERS,
        pin_memory=True,
        persistent_workers=FEATURE_LOADER_WORKERS > 0,
    )
    parts = [extract_selected_representation(batch) for batch in loader]
    payload = {
        'metadata': {
            'cache_signature': TEST_CACHE_SIGNATURE,
            'start': start,
            'stop': stop,
            'ids_signature': sha256_json(test_df.iloc[start:stop]['image_id'].tolist()),
        },
        'row_indices': torch.arange(start, stop, dtype=torch.int64),
        'features': torch.cat(parts),
    }
    atomic_torch_save(payload, path)
    validate_test_shard(path, start, stop)
    del payload, parts, loader, subset

test_features = torch.empty((len(test_df), *TEST_FEATURE_SHAPE), dtype=torch.float16)
test_shard_manifest = []
for start, stop in test_ranges:
    path = test_shard_path(start, stop)
    payload = validate_test_shard(path, start, stop)
    test_features[start:stop].copy_(payload['features'])
    test_shard_manifest.append({
        'file': path.name,
        'start': start,
        'stop': stop,
        'bytes': path.stat().st_size,
        'sha256': sha256_file(path),
    })
    del payload
TEST_CACHE_MANIFEST = {
    'cache_identity': TEST_CACHE_IDENTITY,
    'cache_signature': TEST_CACHE_SIGNATURE,
    'rows': len(test_df),
    'feature_shape': list(test_features.shape),
    'shards': test_shard_manifest,
    'complete': True,
}
write_json_atomic(TEST_CACHE_MANIFEST, ARTIFACT_DIR / 'cache_manifest_test.json')


def predict_feature_tensor(model, candidate_name, features):
    loader = DataLoader(torch.utils.data.TensorDataset(features), batch_size=PREDICT_BATCH_SIZE, shuffle=False,
                        num_workers=0, pin_memory=True)
    parts = []
    model.eval()
    with torch.inference_mode():
        for (batch,) in loader:
            logits = model(batch.to(HEAD_DEVICE, dtype=torch.float32, non_blocking=True))
            probability = logits.softmax(dim=1)
            if not torch.isfinite(probability).all():
                raise RuntimeError('Test inference produced NaN or Inf.')
            parts.append(probability.cpu())
    result = torch.cat(parts).numpy().astype(np.float32)
    if result.shape != (len(features), NUM_CLASSES):
        raise RuntimeError('Test probability shape failed.')
    return result


test_inference_started = time.perf_counter()
test_probabilities = predict_feature_tensor(FINAL_MODEL, SELECTED_CANDIDATE, test_features)
TEST_INFERENCE_SECONDS = time.perf_counter() - test_inference_started
write_npy_atomic(test_probabilities, ARTIFACT_DIR / 'test_probabilities.npy')
test_predictions = test_probabilities.argmax(axis=1).astype(np.int64)
submission = pd.DataFrame({'id': submission_template['id'].to_numpy(), 'predicted': test_predictions})
SUBMISSION_CHECKS = {
    'row_count': len(submission) == EXPECTED_TEST_ROWS,
    'exact_columns': tuple(submission.columns) == SUBMISSION_COLUMNS,
    'no_missing_values': not submission.isna().any().any(),
    'unique_ids': submission['id'].is_unique,
    'labels_are_0_1_2': submission['predicted'].isin([0, 1, 2]).all(),
    'template_row_order': submission['id'].tolist() == submission_template['id'].tolist(),
}
if not all(SUBMISSION_CHECKS.values()):
    raise RuntimeError('Submission validation failed: {}.'.format(
        [name for name, passed in SUBMISSION_CHECKS.items() if not passed]
    ))


def safe_name(value):
    cleaned = ''.join(character if character.isalnum() or character in {'-', '_'} else '_' for character in str(value))
    return cleaned.strip('_') or 'CHANGE_ME'


SUBMISSION_PATH = ARTIFACT_DIR / 'submission_{}.csv'.format(safe_name(TEAM_NAME))
write_csv_atomic(submission, SUBMISSION_PATH)
TEST_INFERENCE_MANIFEST = {
    'selected_candidate': SELECTED_CANDIDATE,
    'selection_signature': SELECTION_SIGNATURE,
    'locked_recipe_signature': LOCKED_RECIPE_SIGNATURE,
    'model_revision': RESOLVED_REVISION,
    'final_head_sha256': FINAL_HEAD_SHA256,
    'test_cache_signature': TEST_CACHE_SIGNATURE,
    'test_order_signature': TEST_ORDER_SIGNATURE,
    'test_dir': str(TEST_DIR),
    'template_path': str(TEMPLATE_PATH),
    'view': 'original',
    'backbone_passes_per_image': 1,
    'head_inference_passes': 1,
    'tta': False,
    'multi_crop': False,
    'horizontal_flip': False,
    'fold_or_seed_ensemble': False,
    'probability_blend': False,
    'specialist_routing': False,
    'threshold_or_class_bias': False,
    'manual_row_correction': False,
    'decision_rule': 'direct argmax',
    'probability_shape': list(test_probabilities.shape),
    'probabilities_finite': bool(np.isfinite(test_probabilities).all()),
    'inference_seconds': TEST_INFERENCE_SECONDS,
    'submission': SUBMISSION_PATH.name,
    'submission_checks': SUBMISSION_CHECKS,
    'predicted_class_counts': {
        str(index): int((test_predictions == index).sum()) for index in range(NUM_CLASSES)
    },
    'previous_submissions_or_leaderboard_used': False,
}
write_json_atomic(TEST_INFERENCE_MANIFEST, ARTIFACT_DIR / 'test_inference_manifest.json')

# Independent reload from persisted projection/head/test cache.
del FINAL_MODEL
release_cuda_cache()
reloaded_projection = torch.load(PROJECTION_STATE_PATH, map_location='cpu', weights_only=False)
reloaded_projection_checksum = projection_checksum(reloaded_projection['matrix'])
reloaded_checkpoint = torch.load(FINAL_HEAD_PATH, map_location='cpu', weights_only=False)
reloaded_test_cache_manifest = read_json(ARTIFACT_DIR / 'cache_manifest_test.json')
reloaded_metadata = reloaded_checkpoint['metadata']
metadata_checks = {
    'candidate_name': reloaded_metadata['candidate'] == SELECTED_CANDIDATE,
    'architecture_name': reloaded_metadata['architecture_name'] == SELECTED_CANDIDATE,
    'representation_name': reloaded_metadata['representation_name'] == HEAD_SPECS[SELECTED_CANDIDATE]['representation'],
    'branch_or_token_names': reloaded_metadata['branch_or_token_names'] == FINAL_HEAD_METADATA['branch_or_token_names'],
    'selection_signature': reloaded_metadata['selection_signature'] == SELECTION_SIGNATURE,
    'train_cache_signature': reloaded_metadata['train_cache_signature'] == TRAIN_CACHE_SIGNATURE,
    'test_cache_signature': bool(
        reloaded_test_cache_manifest.get('cache_signature') == TEST_CACHE_SIGNATURE
        and sha256_json(reloaded_test_cache_manifest.get('cache_identity')) == TEST_CACHE_SIGNATURE
    ),
    'projection_state_checksum': reloaded_projection_checksum == PROJECTION_CHECKSUM,
    'projection_state_metadata': bool(
        reloaded_projection.get('metadata', {}).get('seed') == PROJECTION_SEED
        and reloaded_projection.get('metadata', {}).get('input_dim') == EXPECTED_HIDDEN_SIZE
        and reloaded_projection.get('metadata', {}).get('output_dim') == PROJECTION_DIM
    ),
    'selected_projection_contract': (
        reloaded_metadata['projection_checksum'] == PROJECTION_CHECKSUM
        if SELECTED_CANDIDATE == CANDIDATE_NAME
        else reloaded_metadata['projection_checksum'] is None
    ),
}
if not all(metadata_checks.values()):
    raise RuntimeError('Checkpoint reconstruction metadata failed: {}.'.format(
        [name for name, passed in metadata_checks.items() if not passed]
    ))
RELOADED_MODEL = build_head(SELECTED_CANDIDATE).to(HEAD_DEVICE)
RELOADED_MODEL.load_state_dict(reloaded_checkpoint['state_dict'], strict=True)
reproduced_probabilities = predict_feature_tensor(RELOADED_MODEL, SELECTED_CANDIDATE, test_features)
persisted_test_probabilities = np.load(ARTIFACT_DIR / 'test_probabilities.npy', allow_pickle=False)
if not np.array_equal(persisted_test_probabilities, test_probabilities):
    raise RuntimeError('Persisted test probability file differs from the just-computed tensor.')
persisted_submission = pd.read_csv(SUBMISSION_PATH, dtype={'id': str})
MAX_PROBABILITY_DIFFERENCE = float(np.max(np.abs(
    persisted_test_probabilities - reproduced_probabilities
)))
REPRODUCTION_CHECK = {
    'passed': bool(
        all(metadata_checks.values())
        and np.array_equal(persisted_test_probabilities.argmax(axis=1), reproduced_probabilities.argmax(axis=1))
        and MAX_PROBABILITY_DIFFERENCE <= 1e-6
        and persisted_submission['id'].tolist() == submission_template['id'].tolist()
        and np.array_equal(persisted_submission['predicted'].to_numpy(), test_predictions)
    ),
    'metadata_checks': metadata_checks,
    'persisted_probability_file_matches_initial_inference': True,
    'probabilities_identical': bool(np.array_equal(persisted_test_probabilities, reproduced_probabilities)),
    'predictions_identical': bool(np.array_equal(
        persisted_test_probabilities.argmax(axis=1), reproduced_probabilities.argmax(axis=1)
    )),
    'submission_row_order_identical': (
        persisted_submission['id'].tolist() == submission_template['id'].tolist()
    ),
    'persisted_submission_predictions_identical': bool(np.array_equal(
        persisted_submission['predicted'].to_numpy(), test_predictions
    )),
    'max_absolute_probability_difference': MAX_PROBABILITY_DIFFERENCE,
    'tolerance': 1e-6,
    'projection_required_by_selected_representation': SELECTED_CANDIDATE == CANDIDATE_NAME,
    'projection_state_checksum': reloaded_projection_checksum,
}
if not REPRODUCTION_CHECK['passed']:
    raise RuntimeError('Checkpoint reproduction failed: {}'.format(REPRODUCTION_CHECK))
write_json_atomic(REPRODUCTION_CHECK, ARTIFACT_DIR / 'checkpoint_reproduction_check.json')

REPRODUCIBILITY_MANIFEST = {
    'experiment': EXPERIMENT_NAME,
    'revision_marker': REVISION_MARKER,
    'model_repository': MODEL_REPO_ID,
    'model_revision': RESOLVED_REVISION,
    'config_signature': CONFIG_SIGNATURE,
    'data_contract_signature': DATA_CONTRACT_SIGNATURE,
    'fold_assignment_signature': FOLD_ASSIGNMENT_SIGNATURE,
    'projection_checksum': PROJECTION_CHECKSUM,
    'projection_state_sha256': sha256_file(PROJECTION_STATE_PATH),
    'train_cache_signature': TRAIN_CACHE_SIGNATURE,
    'test_cache_signature': TEST_CACHE_SIGNATURE,
    'selection_signature': SELECTION_SIGNATURE,
    'locked_recipe_signature': LOCKED_RECIPE_SIGNATURE,
    'final_head_sha256': FINAL_HEAD_SHA256,
    'seed': SEED,
    'deterministic_algorithms': True,
    'environment': ENVIRONMENT,
    'single_model_contract': {
        'one_frozen_backbone': True,
        'one_selected_representation': True,
        'one_final_head': True,
        'one_original_test_view': True,
        'one_probability_tensor_and_argmax': True,
    },
    'checkpoint_reproduction': REPRODUCTION_CHECK,
}
write_json_atomic(REPRODUCIBILITY_MANIFEST, ARTIFACT_DIR / 'reproducibility_manifest.json')

selected_oof = OOF_METRICS[SELECTED_CANDIDATE]
FINAL_RUN_SUMMARY = {
    'status': 'complete',
    'selected_candidate': SELECTED_CANDIDATE,
    'candidate_adopted': CANDIDATE_ADOPTED,
    'v11_selected_train_only_oof': {
        'macro_f1': selected_oof['macro_f1'],
        'total_errors': selected_oof['total_errors'],
        'recyclable_organic_errors': selected_oof['ro_errors'],
    },
    'v11_candidate_train_only_oof': (
        None if CANDIDATE_NAME not in OOF_METRICS else {
            'macro_f1': OOF_METRICS[CANDIDATE_NAME]['macro_f1'],
            'total_errors': OOF_METRICS[CANDIDATE_NAME]['total_errors'],
            'recyclable_organic_errors': OOF_METRICS[CANDIDATE_NAME]['ro_errors'],
        }
    ),
    'matched_v11_control_train_only_oof': {
        'macro_f1': OOF_METRICS[CONTROL_NAME]['macro_f1'],
        'total_errors': OOF_METRICS[CONTROL_NAME]['total_errors'],
        'recyclable_organic_errors': OOF_METRICS[CONTROL_NAME]['ro_errors'],
    },
    'historical_v10_control_train_only': V10_REFERENCE,
    'historical_v7_train_only': V7_REFERENCE,
    'selection_gate': SELECTION_DECISION,
    'final_epochs': FINAL_EPOCHS,
    'fold_best_epochs': SELECTED_BEST_EPOCHS,
    'test_probability_shape': list(test_probabilities.shape),
    'submission': SUBMISSION_PATH.name,
    'artifact_bundle': 'artifact_bundle.zip',
    'checkpoint_reproduction_passed': REPRODUCTION_CHECK['passed'],
    'test_used_for_selection': False,
    'external_data_used': False,
    'tta_or_ensemble_used': False,
    'runtime_seconds': time.perf_counter() - NOTEBOOK_STARTED_AT,
}
write_json_atomic(FINAL_RUN_SUMMARY, ARTIFACT_DIR / 'final_run_summary.json')

required_artifacts = [
    'config.json', 'environment.json', 'data_contract.json', 'fold_assignments.csv', 'fold_signature.json',
    'model_manifest.json', 'planned_device_map.json', 'device_map.json', 'preflight_report.json',
    'projection_state.pt', 'projection_manifest.json', 'cache_manifest_train.json', 'cache_manifest_test.json',
    'development_metrics.csv', 'development_metrics.json', 'locked_recipe_before_assessment.json',
    'assessment_per_fold.csv', 'assessment_aggregate.csv', 'selection_decision.json', 'error_analysis.json',
    'query_attention_diagnostics.json', 'query_attention_diagnostics.csv',
    'query_attention_maps_by_true_class.csv',
    'development_predictions_' + CONTROL_NAME + '.csv',
    'development_probabilities_' + CONTROL_NAME + '.npy',
    'development_predictions_' + CANDIDATE_NAME + '.csv',
    'development_probabilities_' + CANDIDATE_NAME + '.npy',
    'final_head.pt', 'final_training_history.csv', 'test_probabilities.npy', SUBMISSION_PATH.name,
    'test_inference_manifest.json', 'reproducibility_manifest.json', 'final_run_summary.json',
    'checkpoint_reproduction_check.json',
]
for candidate_name in OOF_METRICS:
    required_artifacts.extend([
        'oof_predictions_' + candidate_name + '.csv',
        'oof_probabilities_' + candidate_name + '.npy',
        'oof_metrics_' + candidate_name + '.json',
        'confusion_matrix_raw_' + candidate_name + '.csv',
        'confusion_matrix_normalized_' + candidate_name + '.csv',
        'classification_report_' + candidate_name + '.csv',
        'oof_recyclable_organic_errors_' + candidate_name + '.csv',
    ])
missing_artifacts = [name for name in required_artifacts if not (ARTIFACT_DIR / name).is_file()]
if missing_artifacts:
    raise RuntimeError('Required artifacts are missing before bundling: {}.'.format(missing_artifacts))

BUNDLE_PATH = ARTIFACT_DIR / 'artifact_bundle.zip'
temporary_bundle = BUNDLE_PATH.with_name(BUNDLE_PATH.name + '.tmp')
with zipfile.ZipFile(temporary_bundle, 'w', compression=zipfile.ZIP_DEFLATED, compresslevel=6) as archive:
    for path in sorted(ARTIFACT_DIR.rglob('*')):
        if not path.is_file() or path == temporary_bundle or path == BUNDLE_PATH:
            continue
        if '.tmp' in path.name:
            continue
        relative = path.relative_to(ARTIFACT_DIR)
        if relative.parts and relative.parts[0] == 'feature_cache':
            continue
        archive.write(path, relative.as_posix())
os.replace(temporary_bundle, BUNDLE_PATH)
with zipfile.ZipFile(BUNDLE_PATH, 'r') as archive:
    if any(name.startswith('feature_cache/') for name in archive.namelist()):
        raise RuntimeError('Artifact bundle unexpectedly contains feature-cache files.')

print('Run complete:', SELECTED_CANDIDATE)
print('Submission:', SUBMISSION_PATH)
print('Artifact bundle:', BUNDLE_PATH)
if TEAM_NAME == 'CHANGE_ME':
    print('WARNING: set TEAM_NAME and rerun the post-lock cell before submitting.')


## 12. Completion contract

A successful Run All ends with a persisted train-only selection, one full-data head, one original-view test probability tensor, a template-order submission, an independently reloaded checkpoint reproduction check, a reproducibility manifest, a historical comparison summary, and `artifact_bundle.zip` without feature caches. A failed candidate is reported as rejected; gates are never overridden.